# 🔎 17 · Bayesian networks III: inference and learning a signalling network
### Enumeration, variable elimination and sampling on the alarm network; maximum-likelihood CPTs, Chow–Liu trees and BIC hill climbing on simulated single-cell signalling data with a known true graph — and what observational data cannot tell you

**BiologyToML** · Level 1 · 02-620 core · Part D — Probabilistic graphical models · Module 17 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/17_bayesian_networks_inference_learning.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib (graphs drawn with the toolkit's `draw_graph`) |
| **Prerequisites** | Modules 15–16 (Bayesian networks, d-separation), 05 (MLE and MAP) and 08 (estimating probabilities by counting) |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Bayesian networks (part 3, inference and learning): full, partial and conditional queries; query, evidence and hidden variables; inference by enumeration and its cost; variable elimination (19 → 14 operations), polytrees and complexity; stochastic inference by ancestral sampling and counting; learning the conditional probabilities of a known graph by maximum likelihood; structure learning — Chow–Liu trees (KL divergence, mutual information, Kruskal's maximum spanning tree) and hill climbing (search operators, local maxima, random restarts, TABU, annealing); what you should know about Bayesian networks |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Sections 1–7 use the classic alarm network of the 02-620 lecture and data sampled from it (SIMULATED).
> Sections 8–10 use SIMULATED single-cell signalling data shaped like the flow-cytometry study of Sachs et al.
> (2005): 1,800 unperturbed cells, 11 signalling molecules discretised into low/medium/high. They are generated
> from the 20-edge consensus network, which is therefore the true graph that the learning methods try to recover.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Distinguish** query, evidence and hidden variables and **compute** full, partial and conditional probabilities by enumeration ($0.005355$, $P(B,J,\neg M)$, $P(B\mid J,\neg M)$), counting the operations (19).
2. **Implement** variable elimination with factors (product, sum-out, evidence), **reproduce** the 19 → 14 operation count of the worked example, and **explain** why its cost depends on the graph (polytrees, loops, NP-hardness).
3. **Implement** ancestral sampling and the rejection estimate $N_B/N_c$, and **quantify** how the error and the wasted samples depend on $N$ and on how rare the evidence is.
4. **Derive** the maximum-likelihood CPTs from the decomposed log-likelihood (conditional counts), **add** pseudo-counts (MAP), and **measure** how fast learned CPTs converge.
5. **Explain** why structure learning is hard (count the DAGs), **derive** the Chow–Liu result (KL divergence ↔ sum of mutual informations) and **implement** Kruskal's maximum spanning tree on a seven-node worked example.
6. **Learn** a tree and a general DAG (Chow–Liu; hill climbing with BIC and random restarts) from simulated single-cell data and **score** them against the true signalling network (precision and recall of edges and arrow directions).
7. **State** what observational data cannot identify (Markov equivalence, arrow directions), why Sachs et al. needed interventions, and why a learned edge is not a proven mechanism.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the alarm network | 5 min |
| 1 · Queries on the alarm network | full, partial and conditional probabilities | 12 min |
| 2 · Inference by enumeration | a general algorithm; counting operations | 12 min |
| 3 · Variable elimination | factors; 19 → 14; chains; polytrees and NP-hardness | 25 min |
| 4 · Stochastic inference | ancestral sampling, $N_B/N_c$, rare evidence | 18 min |
| 5 · Learning CPTs with a known graph | MLE = conditional counts; MAP; error vs N | 18 min |
| 6 · Why structure learning is hard | Robinson's count of DAGs | 8 min |
| 7 · Chow–Liu trees | KL, mutual information, Kruskal; a seven-node worked example | 24 min |
| 8 · Signalling data: a Chow–Liu tree | SIMULATED Sachs-like cells; 🧪 precision and recall against the true graph | 18 min |
| 9 · Score-based search: hill climbing with BIC | add/delete/reverse; restarts; local maxima; 🧪 vs the truth | 24 min |
| 10 · What observational data cannot tell you | Markov equivalence; interventions | 8 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 8 min |

## 🧫 The biological question

Sachs *et al.* (2005) measured 11 signalling molecules (nine phosphoproteins and the phospholipids PIP2 and PIP3; we
call them all "proteins" for brevity) in thousands of individual human T cells, under several stimulation conditions
and with targeted perturbations (inhibitors or activators of particular molecules). From these measurements they
reconstructed a Bayesian network of the signalling pathway that agreed with most of the connections already known
from biochemistry. Their study is a landmark of **learning biological networks from
data**, and it raises two questions this notebook answers:

1. **Inference.** Given a network and its probability tables, how do we answer questions such as "if PKC is high and Mek
   is low in a cell, how likely is Raf to be high?" — when the answer requires summing over all the unmeasured
   proteins? (Enumeration, variable elimination, sampling.)
2. **Learning.** Given only measurements, how do we estimate the probability tables (easy when the graph is known) and
   the graph itself (hard)? (Maximum likelihood, Chow–Liu trees, score-based search.)

We first learn the methods on the five-variable alarm network, where every answer can be checked exactly, and then
apply them to single-cell signalling data. The repository stores no data, so we use **simulated** cells shaped like
the Sachs measurements (`simulate_signaling_cytometry`). They are generated from the consensus network, so — unlike
with real cells, where that network is only a literature consensus — we know the **true graph** and can grade every
learned edge. The last section is the most important for a biologist: it explains which aspects of a network purely
observational data **cannot** reveal, and why Sachs *et al.* needed interventions. This notebook teaches the third part
of the 02-620 lecture on Bayesian networks.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
from fractions import Fraction
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They provide the classic alarm network, the Sachs et al. 2005 consensus network with a simulator of single-cell data generated from it, precision/recall for comparing learned and true edges, and the graph-drawing function. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ALARM_PARENTS = {"B": (), "E": (), "A": ("B", "E"), "J": ("A",), "M": ("A",)}


ALARM_CPTS = {
    "B": {(): 0.05},
    "E": {(): 0.10},
    "A": {(1, 1): 0.95, (1, 0): 0.85, (0, 1): 0.50, (0, 0): 0.05},
    "J": {(1,): 0.70, (0,): 0.05},
    "M": {(1,): 0.80, (0,): 0.15},
}


SACHS_PROTEINS = ["raf", "mek", "plc", "pip2", "pip3", "erk", "akt", "pka", "pkc", "p38", "jnk"]


SACHS_CONSENSUS_EDGES = [
    ("pka", "raf"), ("pkc", "raf"), ("raf", "mek"), ("pka", "mek"), ("pkc", "mek"),
    ("pip3", "plc"), ("plc", "pip2"), ("pip3", "pip2"), ("mek", "erk"), ("pka", "erk"),
    ("pip3", "akt"), ("erk", "akt"), ("pka", "akt"), ("pkc", "pka"), ("plc", "pkc"),
    ("pip2", "pkc"), ("pka", "p38"), ("pkc", "p38"), ("pka", "jnk"), ("pkc", "jnk"),
]


def simulate_signaling_cytometry(kind: str = "continuous", observational_only: bool = True,
                                 seed: int = 2005) -> dict:
    """Synthetic single-cell phospho-protein data shaped like the Sachs et al. (2005) signalling study.

    11 signalling proteins of human T cells, generated by a linear-Gaussian structural model on
    the 20-edge consensus network ``SACHS_CONSENSUS_EDGES`` (log intensities).  Besides
    unperturbed cells ("none"), some cells receive interventions that set one protein high
    (activator) or low (inhibitor): pkc (both), akt, pip2 and mek (inhibitors), pka (activator).
    ``kind='continuous'``: 7,466 cells; ``'discrete'``: 5,400 cells, each protein cut into
    low/medium/high (0/1/2) at the tertiles of the unperturbed cells.  The condition sizes follow
    the common benchmark encoding of the original data, in which the 2,603 "none" cells are the
    two general-stimulation conditions (853 + 902 cells) plus the 848 PI3K-inhibitor cells.  With
    ``observational_only=True`` only unperturbed cells are returned.  The two kinds are drawn
    with different seeds, so their edge strengths differ.

    Returns a dict with ``data`` (DataFrame, 11 proteins), ``intervention``, ``edges`` (the true
    network), ``weights`` (true edge strengths) and ``source``.
    """
    rng = np.random.default_rng(seed if kind == "continuous" else seed + 1)
    order = ["pip3", "plc", "pip2", "pkc", "pka", "raf", "mek", "erk", "akt", "p38", "jnk"]   # topological
    sign = {("pip3", "pip2"): -1, ("plc", "pip2"): -1}                   # PIP2 is consumed
    weights = {e: sign.get(e, 1) * rng.uniform(0.35, 0.85) for e in SACHS_CONSENSUS_EDGES}
    base = dict(zip(order, rng.uniform(2.5, 5.0, len(order))))
    noise = dict(zip(order, rng.uniform(0.45, 0.8, len(order))))
    sizes = ({"none": 2603, "pkc": 1636, "akt": 911, "pip2": 810, "mek": 799, "pka": 707} if kind == "continuous"
             else {"none": 1800, "pkc": 1200, "pka": 600, "akt": 600, "pip2": 600, "mek": 600})
    frames, labels = [], []
    for target, n in sizes.items():
        dev = {}
        for v in order:
            if v == target:                                              # intervention: do(v = high/low)
                direction = (np.where(rng.random(n) < 0.5, 1.0, -1.0) if v == "pkc"
                             else np.full(n, 1.0 if v == "pka" else -1.0))
                dev[v] = 1.6 * direction + rng.normal(0, 0.3, n)
            else:
                dev[v] = sum(w * dev[p] for (p, c), w in weights.items() if c == v) + rng.normal(0, noise[v], n)
        frames.append(pd.DataFrame({v: np.round(base[v] + dev[v], 4) for v in SACHS_PROTEINS}))
        labels += [target] * n
    data = pd.concat(frames, ignore_index=True)
    ivn = np.array(labels, dtype=object)
    if kind == "discrete":
        ref = data[ivn == "none"]
        for v in SACHS_PROTEINS:
            cuts = np.quantile(ref[v], [1 / 3, 2 / 3])
            data[v] = np.searchsorted(cuts, data[v].to_numpy()).astype(int)
    if observational_only:
        keep = ivn == "none"
        data, ivn = data[keep].reset_index(drop=True), ivn[keep]
    return dict(data=data, intervention=ivn, edges=list(SACHS_CONSENSUS_EDGES),
                weights=pd.Series({f"{p}→{c}": w for (p, c), w in weights.items()}),
                source="SIMULATED signalling data (shaped like Sachs et al. 2005)")


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


def precision_recall_f1(y_true, y_pred):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 (0 when undefined)."""
    c = confusion_counts(y_true, y_pred)
    precision = c["TP"] / (c["TP"] + c["FP"]) if c["TP"] + c["FP"] else 0.0
    recall = c["TP"] / (c["TP"] + c["FN"]) if c["TP"] + c["FN"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_signaling_cytometry', 'confusion_counts', 'precision_recall_f1', 'draw_graph']))

In [ ]:
# ── 0.1 The alarm network and the exact joint probability (Modules 15–16) ──
ALARM_POS = {"B": (0.2, 0.85), "E": (0.8, 0.85), "A": (0.5, 0.5), "J": (0.25, 0.15), "M": (0.75, 0.15)}
ALARM_NODES = list(ALARM_PARENTS)


def edges_from_parents(parents):
    """Convert a parents dictionary into a list of (parent, child) edges."""
    return [(p, child) for child, pa in parents.items() for p in pa]


def cpt_prob(node, value, assignment, parents, cpts):
    """P(node = value | the values its parents take in `assignment`), from an ALARM-style CPT."""
    p1 = cpts[node][tuple(assignment[p] for p in parents[node])]
    return p1 if value == 1 else 1 - p1


def joint_prob(assignment, parents, cpts):
    """P(complete assignment) = product over nodes of P(x_i | pa(x_i))."""
    prob = 1.0
    for node in parents:
        prob *= cpt_prob(node, assignment[node], assignment, parents, cpts)
    return prob


print("edges:", edges_from_parents(ALARM_PARENTS))

---
## 1 · Queries on the alarm network

> 📍 **Where we are.** Module 15 computed the probability of one *complete* configuration; Module 16 read
> independences off the graph. Now we compute **numbers** for questions that involve only *some* of the variables
> Notation as before: $B$ means $B=1$, $\neg B$ means $B=0$.

**Inference on the distribution.** Module 16 answered independence questions from the graph alone. Now we compute
probabilities, with three methods of increasing speed: **enumeration** (exact, slow), **variable elimination** (exact,
faster) and **stochastic inference** (approximate, fast). The CPTs are the classic ones: $P(B)=0.05$, $P(E)=0.1$,
$P(A\mid B,E)=0.95$, $P(A\mid B,\neg E)=0.85$, $P(A\mid\neg B,E)=0.5$, $P(A\mid\neg B,\neg E)=0.05$, $P(J\mid A)=0.7$,
$P(J\mid\neg A)=0.05$, $P(M\mid A)=0.8$, $P(M\mid\neg A)=0.15$.

**Full joint.** "There was a burglary, no earthquake, the alarm went off, John called, Mary did not":

$$P(B,\neg E,A,J,\neg M)=P(B)\,P(\neg E)\,P(A\mid B,\neg E)\,P(J\mid A)\,P(\neg M\mid A)=0.05\times0.9\times0.85\times0.7\times0.2 .$$

*In words:* one look-up per node, multiplied together. We can easily compute a complete joint distribution. What about
*partial* distributions? *Conditional* distributions?

**Partial distribution.** "There was a burglary, John called, Mary did not call" says nothing about
$E$ and $A$. Marginalisation, $P(X)=\sum_y P(X,y)$, sums them out:

$$P(B,J,\neg M)=\sum_{a\in\{0,1\}}\sum_{e\in\{0,1\}}P(B,J,\neg M,A{=}a,E{=}e).$$

*In words:* add up the probabilities of the four complete worlds that agree with the statement — one for each way the
unmentioned variables could have turned out.

In [ ]:
# ── 1.1 Full and partial joint probabilities ──
full = joint_prob({"B": 1, "E": 0, "A": 1, "J": 1, "M": 0}, ALARM_PARENTS, ALARM_CPTS)
print(f"P(B,¬E,A,J,¬M) = {full:.6f}\n")
terms = {}
for a, e in [(1, 1), (0, 1), (1, 0), (0, 0)]:                      # the order of the hand calculation
    terms[(a, e)] = joint_prob({"B": 1, "J": 1, "M": 0, "A": a, "E": e}, ALARM_PARENTS, ALARM_CPTS)
    print(f"P(B, J, ¬M, {'A' if a else '¬A'}, {'E' if e else '¬E'}) = {terms[(a, e)]:.7f}")
p_B_J_notM = sum(terms.values())
print(f"P(B, J, ¬M) = {p_B_J_notM:.7f}   (hand calculation rounding each term: 0.0007 + 0.00001 + 0.005 + 0.0003 = 0.00601)")

🔍 **What just happened?** The full joint reproduces 0.005355. The four terms of the partial distribution are 0.000665,
0.0000106, 0.005355 and 0.000287, and they add up to **0.0063175**. A quick hand calculation that rounds each term
before adding (0.0007 + 0.00001 + 0.005 + 0.0003) gives 0.00601; the difference comes almost entirely from rounding
0.005355 down to 0.005. In this notebook we always keep full precision.

**Conditional distribution.** "Given that John called and Mary did not, was there a burglary?" By the
definition of conditional probability,

$$P(B\mid J,\neg M)=\frac{P(B,J,\neg M)}{P(J,\neg M)}=\frac{P(B,J,\neg M)}{P(B,J,\neg M)+P(\neg B,J,\neg M)} .$$

*In words:* compute the partial probability for both values of the query variable and divide one by their sum — the
**normalisation** step. Three roles: **query** variable $B$ (what we ask about),
**evidence** variables $J, M$ (what we observed), **hidden** variables $A, E$ (everything else, summed out).

In [ ]:
# ── 1.2 The conditional probability P(B | J, ¬M) by normalisation ──
p_notB_J_notM = sum(joint_prob({"B": 0, "J": 1, "M": 0, "A": a, "E": e}, ALARM_PARENTS, ALARM_CPTS)
                    for a in (0, 1) for e in (0, 1))
p_J_notM = p_B_J_notM + p_notB_J_notM
P_B_GIVEN_J_NOTM = p_B_J_notM / p_J_notM
print(f"P(¬B, J, ¬M) = {p_notB_J_notM:.6f}")
print(f"P(J, ¬M)     = {p_J_notM:.6f}")
print(f"P(B | J, ¬M) = {p_B_J_notM:.6f} / {p_J_notM:.6f} = {P_B_GIVEN_J_NOTM:.4f}   (prior P(B) = 0.05)")

🔍 **What just happened?** John's call together with Mary's silence raises the probability of a burglary from 5 % to
**11.4 %**. Mary's silence pulls in the opposite direction from John's call: with $J$ alone the probability would be 22 %
(Module 16). Note that the evidence itself is not rare: on about 5.5 % of days John calls and Mary does not.

✏️ **Your turn.** Compute $P(B\mid J, M)$ (both neighbours call) with the same two-line recipe. Before running, predict
whether it is larger or smaller than 0.114. Then try one more query of each kind: the partial distributions $P(A)$ and $P(M,E)$, and the
conditional distributions $P(J\mid B,E)$ and $P(A\mid B,M)$ — which variables are hidden in each?

### 📘 Lecture notes — Full, partial and conditional queries

- **Full joint** (all variables specified): one product of CPT entries,
  $P(B,\neg E,A,J,\neg M)=P(B)P(\neg E)P(A\mid B,\neg E)P(J\mid A)P(\neg M\mid A)=0.05\times0.9\times0.85\times0.7\times0.2=0.005355$.
- **Partial** (some variables unspecified): sum the full joint over the unspecified ones,
  $$P(B,J,\neg M)=\sum_{a}\sum_{e}P(B,J,\neg M,A{=}a,E{=}e)=0.0063175 .$$
  *In words:* add the probabilities of all complete worlds that agree with the statement (four here).
- **Conditional:** $$P(B\mid J,\neg M)=\frac{P(B,J,\neg M)}{P(J,\neg M)}=\frac{P(B,J,\neg M)}{P(B,J,\neg M)+P(\neg B,J,\neg M)}=\frac{0.0063175}{0.0555}=0.114 .$$
  *In words:* compute the partial probability for each value of the query and normalise.
- **Roles:** query variable $B$; evidence variables $J, M$; hidden variables $A, E$ (summed out).
- **The catch.** The number of terms is exponential in the number of hidden variables; reusing computations helps
  (Section 3).

> 🎓 **Exam tip.** Never round intermediate terms in a sum of small probabilities — keep 4 significant digits per term. And the
    denominator $P(J,\neg M)$ is obtained for free by also computing $P(\neg B,J,\neg M)$.

---
## 2 · Inference by enumeration

> 📍 **Where we are.** Section 1 did the sums by hand. Enumeration is the general algorithm: for each value of the
> query variable, add up the joint probability of every assignment of the hidden variables — and count how many
> computations that takes.

$$P(q\mid \mathbf e)\;\propto\;\sum_{\mathbf h}\;\prod_{i=1}^{n}P\big(x_i\mid \mathrm{pa}(x_i)\big)\Big|_{Q=q,\;\mathbf E=\mathbf e,\;\mathbf H=\mathbf h}.$$

*In words:* fix the query value $q$ and the evidence $\mathbf e$; loop over all $2^{|\mathbf H|}$ combinations $\mathbf h$ of
the hidden variables; for each, multiply the $n$ CPT entries; add everything up; finally normalise over $q$.

**Counting operations.** For $P(B,J,\neg M)$ there are 4 terms (two hidden binary variables), each a
product of 5 factors, i.e. 4 multiplications, and the 4 terms need 3 additions: $4\times4+3=19$ operations.

In [ ]:
# ── 2.1 A general enumeration routine that also counts operations ──
def enumerate_query(parents, cpts, query, evidence):
    """P(query | evidence) by summing over all hidden assignments.
    Returns (normalised distribution [P(q=0), P(q=1)], unnormalised values, operation counts per query value)."""
    nodes = list(parents)
    hidden = [v for v in nodes if v != query and v not in evidence]
    unnorm, ops = np.zeros(2), {"mul": 0, "add": 0}
    for q in (0, 1):
        terms = []
        for values in itertools.product([0, 1], repeat=len(hidden)):
            world = {**evidence, query: q, **dict(zip(hidden, values))}
            term = 1.0
            for node in nodes:
                term *= cpt_prob(node, world[node], world, parents, cpts)
            terms.append(term)
        unnorm[q] = sum(terms)
    ops["mul"] = (len(nodes) - 1) * len(terms)        # per query value: n − 1 multiplications per term
    ops["add"] = len(terms) - 1                       # ... and (#terms − 1) additions
    return unnorm / unnorm.sum(), unnorm, ops


dist, unnorm, ops = enumerate_query(ALARM_PARENTS, ALARM_CPTS, "B", {"J": 1, "M": 0})
print(f"P(B=1, J, ¬M) = {unnorm[1]:.7f}   P(B=0, J, ¬M) = {unnorm[0]:.7f}")
print(f"P(B=1 | J, ¬M) = {dist[1]:.4f}")
print(f"operations for one query value: {ops['mul']} multiplications + {ops['add']} additions = {ops['mul'] + ops['add']}")

🔍 **What just happened?** The routine reproduces Section 1 ($P(B\mid J,\neg M)=0.1138$) and the count of 19
operations for one value of the query. The trouble is the factor $2^{|\mathbf H|}$: every additional hidden binary
variable **doubles** the number of terms — the number of possible assignments is exponential in the number of
unobserved variables. With 30 hidden genes there would be a billion terms per query. Section 3 measures this
growth and fixes it by re-using calculations.

✏️ **Your turn.** Use `enumerate_query` to compute $P(E{=}1\mid J{=}1, M{=}1)$ and $P(A{=}1\mid B{=}1)$. How many terms does
each need? Which variables are hidden in each case?

---
## 3 · Variable elimination: reuse computations

> 📍 **Where we are.** Enumeration recomputes the same products again and again. Variable elimination (VE) organises
> the sum so that each intermediate result is computed once and reused ("reuse computations rather than recompute
> probabilities").

**The trick is the distributive law**, $ab+ac=a(b+c)$: a factor that does not depend on a summation variable can be
pulled out of that sum. Step by step:

$$\begin{aligned}
P(B,J,\neg M) &= \sum_e\sum_a P(B)\,P(e)\,P(a\mid B,e)\,P(J\mid a)\,P(\neg M\mid a)\\
&= P(B)\sum_e P(e)\underbrace{\sum_a P(a\mid B,e)\,P(J\mid a)\,P(\neg M\mid a)}_{f_{A,J,M}(B,e)}
 = P(B)\underbrace{\sum_e P(e)\,f_{A,J,M}(B,e)}_{f_{E,A,J,M}(B)} = P(B)\,f_{E,A,J,M}(B).
\end{aligned}$$

*In words:* $P(B)$ does not depend on $e$ or $a$, so it moves outside both sums; $P(e)$ does not depend on $a$, so it
moves outside the inner sum. The inner sum produces a small table $f_{A,J,M}(B,e)$ — a **factor**, a function of $B$
and $e$ with $J, M$ fixed and $a$ summed out — and the outer sum turns it into $f_{E,A,J,M}(B)$.

**Counting.** Inner sum: 2 terms × 2 multiplications + 1 addition = 5 (for each $e$). Next sum: 2 terms
× (1 multiplication + the 5 operations for $f_{A,J,M}$) + 1 addition = 13. One more multiplication by $P(B)$: **14
operations instead of 19.** Let us count them with instrumented arithmetic.

In [ ]:
# ── 3.1 Reproducing 19 → 14 operations with counting arithmetic (B = 1, J = 1, M = 0 fixed) ──
ops = {"mul": 0, "add": 0}


def mul(x, y):
    """Multiply and count."""
    ops["mul"] += 1
    return x * y


def add(x, y):
    """Add and count."""
    ops["add"] += 1
    return x + y


def P(node, value, **given):
    """Short-hand CPT look-up, e.g. P('A', 1, B=1, E=0)."""
    return cpt_prob(node, value, {**given, node: value}, ALARM_PARENTS, ALARM_CPTS)


total = None                                                        # enumeration: Σ_e Σ_a of a 5-factor product
for e in (1, 0):
    for a in (1, 0):
        term = mul(mul(mul(mul(P("B", 1), P("E", e)), P("A", a, B=1, E=e)), P("J", 1, A=a)), P("M", 0, A=a))
        total = term if total is None else add(total, term)
print(f"enumeration:          {total:.7f}  using {ops['mul']} mult + {ops['add']} add = {sum(ops.values())} operations")

ops.update(mul=0, add=0)
outer = None                                                        # variable elimination: P(B) Σ_e P(e) Σ_a (...)
for e in (1, 0):
    f_AJM = None
    for a in (1, 0):
        t = mul(mul(P("A", a, B=1, E=e), P("J", 1, A=a)), P("M", 0, A=a))
        f_AJM = t if f_AJM is None else add(f_AJM, t)                # 2 × 2 mult + 1 add = 5 per e
    term = mul(P("E", e), f_AJM)
    outer = term if outer is None else add(outer, term)             # 2 × (1 + 5) + 1 = 13
result = mul(P("B", 1), outer)                                       # + 1 = 14
print(f"variable elimination: {result:.7f}  using {ops['mul']} mult + {ops['add']} add = {sum(ops.values())} operations")

# the elimination as pictures: evidence J, M shaded; eliminated variables greyed out
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
steps = [("start: 5 factors", [], "P(B) P(E) P(A|B,E)\nP(J=1|A) P(M=0|A)"),
         ("sum out A (5 operations per e)", ["A", "J", "M"], "P(B) P(E) f_AJM(B,e)\nf_AJM = Σ_a P(a|B,e) P(J|a) P(¬M|a)"),
         ("sum out E (13 operations so far)", ["A", "J", "M", "E"], "P(B) f_EAJM(B), 1 more × → 14\nf_EAJM = Σ_e P(e) f_AJM(B,e)")]
for ax, (title, gone, text) in zip(axes, steps):
    keep = {v: ALARM_POS[v] for v in ALARM_NODES if v not in gone}
    draw_graph([e for e in edges_from_parents(ALARM_PARENTS) if e[0] in keep and e[1] in keep], keep, ax=ax,
               highlight=[v for v in ("J", "M") if v in keep])
    for v in gone:                                              # ghosts of eliminated / absorbed variables
        ax.add_patch(plt.Circle(ALARM_POS[v], 0.07, fc="0.95", ec="0.7", ls="--"))
        ax.text(*ALARM_POS[v], v, ha="center", va="center", fontsize=9, color="0.6")
    if gone and "E" not in gone:                                # the new factor links B and E
        ax.plot([0.27, 0.73], [0.85, 0.85], ls="--", color="tab:purple", lw=2)
        ax.text(0.5, 0.9, "f_AJM(B,e)", ha="center", fontsize=8, color="tab:purple")
    ax.text(0.5, -0.08, text, ha="center", va="top", fontsize=8)
    ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Same answer (0.0063175), 19 versus 14 operations — reuse of computation, "19 → 14". The
pictures show the bookkeeping: summing out A absorbs the factors of A, J and M into one new factor that links B and E;
summing out E leaves a single factor of B, which is multiplied by $P(B)$. On five variables the saving is modest. To see VE's real power we need a general implementation and bigger networks.

### 3.1 Factors: the data structure of variable elimination

A **factor** is a table indexed by some variables, e.g. a CPT $P(A\mid B,E)$ is a factor over $(A,B,E)$ with $2^3=8$
entries. VE needs three operations:
- **restrict** a factor to observed evidence (keep only the slice with $J=1$);
- **multiply** two factors (match entries on shared variables) — this creates a factor over the *union* of their
  variables;
- **sum out** a variable (add the slices for its values) — this removes it.

**Algorithm.** Restrict all CPTs to the evidence. For each hidden variable in a chosen **elimination order**: multiply
together all factors that mention it, sum it out, and put the result back. Finally multiply what is left (a factor over
the query) and normalise. The cost is dominated by the *largest factor created*, which depends on the graph and the
order.

In [ ]:
# ── 3.2 A small factor class (binary variables) with operation counters ──
class Factor:
    """A table over binary variables; `variables` names the axes of `table` (each of length 2)."""
    ops = {"mul": 0, "add": 0}                          # shared counters over all factor operations

    def __init__(self, variables, table):
        self.vars, self.table = list(variables), np.asarray(table, float)

    def _aligned(self, all_vars):
        """This table with its axes reordered/expanded to broadcast against `all_vars`."""
        own = [v for v in all_vars if v in self.vars]
        t = np.transpose(self.table, [self.vars.index(v) for v in own])
        return t.reshape([2 if v in self.vars else 1 for v in all_vars])

    def __mul__(self, other):
        all_vars = list(dict.fromkeys(self.vars + other.vars))
        out = self._aligned(all_vars) * other._aligned(all_vars)
        Factor.ops["mul"] += out.size
        return Factor(all_vars, out)

    def sum_out(self, var):
        axis = self.vars.index(var)
        out = self.table.sum(axis=axis)
        Factor.ops["add"] += out.size                     # two values → one addition per output entry
        return Factor([v for v in self.vars if v != var], out)

    def restrict(self, var, value):
        if var not in self.vars:
            return self
        return Factor([v for v in self.vars if v != var], np.take(self.table, value, axis=self.vars.index(var)))

In [ ]:
# ── 3.3 CPTs as factors, and the elimination algorithm ──
def cpt_factor(node, parents, cpts):
    """The CPT of `node` as a factor over (node, *parents): table[x, pa...] = P(node = x | pa)."""
    table = np.zeros([2] * (1 + len(parents[node])))
    for pa_vals, p1 in cpts[node].items():
        table[(1, *pa_vals)], table[(0, *pa_vals)] = p1, 1 - p1
    return Factor([node, *parents[node]], table)


def variable_elimination(parents, cpts, query, evidence, order):
    """P(query | evidence) by eliminating the hidden variables in `order`."""
    factors = []
    for node in parents:
        f = cpt_factor(node, parents, cpts)
        for var, val in evidence.items():
            f = f.restrict(var, val)
        factors.append(f)
    for var in order:                                    # eliminate one hidden variable at a time
        involved = [f for f in factors if var in f.vars]
        factors = [f for f in factors if var not in f.vars]
        prod = involved[0]
        for f in involved[1:]:
            prod = prod * f
        factors.append(prod.sum_out(var))
    result = factors[0]
    for f in factors[1:]:
        result = result * f
    return result.table / result.table.sum()

In [ ]:
# ── 3.4 VE on the alarm network: same answer, any elimination order ──
for order in (["A", "E"], ["E", "A"]):
    Factor.ops.update(mul=0, add=0)
    dist_ve = variable_elimination(ALARM_PARENTS, ALARM_CPTS, "B", {"J": 1, "M": 0}, order)
    print(f"order {order}: P(B=1 | J, ¬M) = {dist_ve[1]:.6f}   factor operations: {Factor.ops}")
print(f"enumeration:     P(B=1 | J, ¬M) = {P_B_GIVEN_J_NOTM:.6f}")

🔍 **What just happened?** VE gives the same 0.113845 for both elimination orders. Its operation counts (28 when A is
eliminated first, 22 when E is eliminated first) differ from the hand count of 14 because the general algorithm
computes the answer for both values of $B$ at once; and they show that the **elimination order** changes the cost even
in this tiny network. What matters most is how the cost **grows** with the network. Storing and re-using intermediate
results in a table is the idea of *dynamic programming* — the factor bookkeeping above is exactly that.

Now a network where enumeration hurts: a **signalling cascade** $X_1\to X_2\to\cdots\to X_n$ (random CPTs), query
$P(X_1\mid X_n{=}1)$ — "the transcription factor at the end is active; was the receptor at the start activated?" All
$n-2$ middle steps are hidden.

In [ ]:
# ── 3.5 Enumeration grows exponentially, variable elimination linearly (cascades of n binary nodes) ──
rng = np.random.default_rng(SEED + 3)
sizes, enum_ops, ve_ops = range(3, 15), [], []
for n in sizes:
    chain = [f"X{i}" for i in range(1, n + 1)]
    par = {v: (() if i == 0 else (chain[i - 1],)) for i, v in enumerate(chain)}
    cp = {v: ({(): 0.5} if i == 0 else {(1,): rng.uniform(0.6, 0.95), (0,): rng.uniform(0.05, 0.4)})
          for i, v in enumerate(chain)}
    d_enum, _, o = enumerate_query(par, cp, "X1", {chain[-1]: 1})
    enum_ops.append(2 * (o["mul"] + o["add"]))                   # both values of the query
    Factor.ops.update(mul=0, add=0)
    d_ve = variable_elimination(par, cp, "X1", {chain[-1]: 1}, order=chain[1:-1][::-1])
    ve_ops.append(Factor.ops["mul"] + Factor.ops["add"])
    assert np.allclose(d_enum, d_ve)
print("n          :", list(sizes))
print("enumeration:", enum_ops)
print("elimination:", ve_ops)

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.semilogy(list(sizes), enum_ops, "o-", color="tab:red", label="enumeration")
ax.semilogy(list(sizes), ve_ops, "s-", color="tab:blue", label="variable elimination")
ax.set_xlabel("length n of the cascade (n − 2 hidden variables)"); ax.set_ylabel("arithmetic operations (log)")
ax.set_title("P(X1 | Xn = 1) in a chain X1 → … → Xn"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both methods give identical probabilities (the `assert` checks it), but enumeration's cost
doubles with every extra node — a straight line on the log scale — while VE grows by a constant 6 operations per node
(one small product and one sum-out per eliminated variable). For $n=14$ enumeration needs 114,686 operations and VE 74.

**When does VE stay cheap?** Re-using computations reduces the running time, but there are still networks in which
the algorithm takes exponential time: its efficiency depends on the network structure. A **polytree** is a DAG in which no two nodes are joined by more than
one path (ignoring directions). On polytrees VE (or the equivalent message-passing algorithm) runs in time linear in the
number of nodes. When the graph has loops, eliminating a variable can create a large factor that links many variables,
and in the worst case the cost is exponential: exact inference in general Bayesian networks is **NP-hard** (Cooper
1990) — it belongs to a family of problems for which no algorithm is known whose running time grows only polynomially
(like $n^2$ or $n^3$) with the size of the network, and computer scientists widely believe none exists. The alarm
network is a polytree; adding an edge $E\to M$ makes it one that is not.

In [ ]:
# ── 3.6 Polytree or not? (union–find on the undirected skeleton) ──
def is_polytree(nodes, edges):
    """True if the skeleton has no undirected cycle (no two nodes joined by two different paths)."""
    root = {v: v for v in nodes}

    def find(v):
        while root[v] != v:
            v = root[v]
        return v

    for u, v in edges:
        ru, rv = find(u), find(v)
        if ru == rv:                     # u and v already connected → this edge closes a loop
            return False
        root[ru] = rv
    return True


alarm_edges = edges_from_parents(ALARM_PARENTS)
print("alarm network        :", is_polytree(ALARM_NODES, alarm_edges))
print("alarm network + E→M  :", is_polytree(ALARM_NODES, alarm_edges + [("E", "M")]))
print("Sachs consensus net  :", is_polytree(SACHS_PROTEINS, SACHS_CONSENSUS_EDGES))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
draw_graph(alarm_edges, ALARM_POS, ax=axes[0], title="polytree: one path between any two nodes")
draw_graph(alarm_edges + [("E", "M")], ALARM_POS, ax=axes[1], highlight=["E", "A", "M"],
           title="not a polytree: E→M closes the loop E–A–M")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The alarm network is a polytree; $E\to M$ creates a second path between E and M (directly,
and via A). The signalling network is far from a polytree — PKA and PKC both act on Raf, Mek,
p38 and JNK, creating many loops — so exact inference on large signalling or regulatory networks quickly needs either
clever elimination orders or approximate methods. That is the motivation for sampling.

✏️ **Your turn.** Is the seven-node graph of Module 16 ($U\to W$, $V\to W$, $V\to X$, $V\to T$, $W\to Y$, $X\to Y$,
$T\to Z$) a polytree? Check with `is_polytree`. If not, which single edge could you delete to make it one?

### 📘 Lecture notes — Enumeration and variable elimination

- **Enumeration.** $P(q\mid\mathbf e)\propto\sum_{\mathbf h}\prod_i P(x_i\mid\mathrm{pa}(x_i))$ over all assignments
  $\mathbf h$ of the hidden variables. For $P(B,J,\neg M)$: 4 terms × 4 multiplications + 3 additions = **19
  operations**. Cost doubles with every hidden binary variable.
- **Variable elimination (VE).** Reuse computations: push each sum as far inside the product as possible
  (distributive law),
  $$P(B,J,\neg M)=P(B)\sum_e P(e)\underbrace{\sum_a P(a\mid B,e)P(J\mid a)P(\neg M\mid a)}_{f_{A,J,M}(B,e)}=P(B)\,f_{E,A,J,M}(B).$$
  *In words:* compute the inner table once per value of $e$, then the outer one, then multiply by $P(B)$.
- **Count.** Inner sum: 2 terms × 2 multiplications + 1 addition = 5; next sum: 2 × (1 + 5) + 1 = 13; one more
  multiplication: **14 operations instead of 19**.
- **Normalisation.** Compute the unnormalised value for each query value and divide by their sum to get
  $P(B\mid J,\neg M)$.
- **Complexity.** VE can save a great deal, but its efficiency depends on the network structure (and the
  elimination order). In a **polytree** (no two nodes joined by more than one path) exact inference is linear in the
  number of nodes; the alarm network is a polytree, the alarm network plus $E\to M$ is not. In general, exact
  inference is **NP-hard**.

> 🎓 **Exam tip.** Be able to rewrite a sum of products with the sums pushed inside as far as possible and to count the
    operations; know that VE's cost is governed by the largest intermediate factor, which depends on the graph and
    the elimination order.

---
## 4 · Stochastic inference: answering questions by sampling

> 📍 **Where we are.** Exact methods can be too slow on large loopy networks. Sampling methods are *inexact but fast*:
> generate many random worlds from the network and count.

**Estimating a probability from samples.** Suppose we can generate samples of $B$ (think of eight coin flips with four
heads: $\hat P(H)=4/8$). With samples $b_1,\dots,b_n$ of $B$,

$$\hat P(B=1)=\frac{1}{n}\sum_{i=1}^{n}\mathbf 1\{b_i=1\}.$$

*In words:* the fraction of samples in which $B=1$. The **indicator** $\mathbf 1\{\cdot\}$ is 1 when the condition holds
and 0 otherwise.

**Conditional probabilities.** With samples of $(B,E)$, $P(B{=}1\mid E{=}1)$ is estimated by the fraction of samples
with $E=1$ that also have $B=1$. In general, count only the samples where the evidence holds:

$$\hat P(B\mid J,\neg M)=\frac{\hat P(B,J,\neg M)}{\hat P(J,\neg M)}=\frac{N_B/N}{N_c/N}=\frac{N_B}{N_c},$$

*In words:* among the $N_c$ samples in which John called and Mary did not, the fraction $N_B/N_c$ that also had a
burglary. This is **rejection sampling**: samples that contradict the evidence are thrown away.

**How to generate joint samples?** One could list all $2^5$ joint probabilities and sample from that table — a bad
idea, because for $n$ variables the table has $2^n$ entries, which is exactly what we want to avoid.
Instead, **ancestral sampling** draws the variables in topological order, each from its CPT given the parents already
drawn: B from $P(B)$, E from $P(E)$, A from $P(A\mid B,E)$, J from $P(J\mid A)$, M from $P(M\mid A)$ (Module 15, Section 6).

In [ ]:
# ── 4.1 Ancestral sampling, step by step and vectorised ──
def ancestral_sample(parents, cpts, n, rng):
    """n joint samples from a binary BN, drawing nodes in the (topological) order of the parents dict."""
    out = {}
    for node, pa in parents.items():                   # the toolkit dict lists parents before children
        p1 = np.empty(n)
        for pa_vals, theta in cpts[node].items():      # P(node = 1 | each sample's parent values)
            rows = np.ones(n, bool)
            for parent, value in zip(pa, pa_vals):
                rows &= out[parent] == value
            p1[rows] = theta
        out[node] = (rng.random(n) < p1).astype(int)
    return pd.DataFrame(out)


rng = np.random.default_rng(SEED + 4)
one = {}
for step, node in enumerate(ALARM_NODES, 1):
    pa_vals = tuple(one[p] for p in ALARM_PARENTS[node])
    theta = ALARM_CPTS[node][pa_vals]
    one[node] = int(rng.random() < theta)
    cond = ",".join(f"{p}={one[p]}" for p in ALARM_PARENTS[node])
    print(f"Step {step}: sample {node} from P({node}{'|' + cond if cond else ''}) (P(=1) = {theta}) → {node} = {one[node]}")

samples = ancestral_sample(ALARM_PARENTS, ALARM_CPTS, 1_000_000, rng)
print(f"\n10^6 samples: P(B=1) ≈ {samples['B'].mean():.4f} (exact 0.05), P(A=1) ≈ {samples['A'].mean():.4f} (exact 0.1333)")

In [ ]:
# ── 4.2 Rejection estimate N_B / N_c of P(B | J, ¬M) and its error versus N ──
def rejection_estimate(df, query, evidence):
    """(N_B / N_c, N_c): the fraction of evidence-consistent samples with query = 1."""
    keep = np.ones(len(df), bool)
    for var, val in evidence.items():
        keep &= df[var].to_numpy() == val
    n_c = keep.sum()
    return (df.loc[keep, query].mean() if n_c else np.nan), n_c


est, n_c = rejection_estimate(samples, "B", {"J": 1, "M": 0})
print(f"N = 10^6: N_c = {n_c:,} kept ({n_c / len(samples):.2%}), estimate {est:.4f}, exact {P_B_GIVEN_J_NOTM:.4f}")

rng = np.random.default_rng(SEED + 42)
Ns, reps = [100, 1_000, 10_000, 100_000], 60
spread = {}
for N in Ns:
    spread[N] = np.array([rejection_estimate(ancestral_sample(ALARM_PARENTS, ALARM_CPTS, N, rng), "B",
                                             {"J": 1, "M": 0})[0] for _ in range(reps)])
    s = spread[N]
    print(f"N = {N:>7,}: mean estimate {np.nanmean(s):.4f}, sd {np.nanstd(s):.4f}, runs with N_c = 0: {np.isnan(s).sum()}")

fig, ax = plt.subplots(figsize=(6, 3.4))
for N in Ns:
    ax.scatter(np.full(reps, N) * np.exp(rng.normal(0, 0.06, reps)), spread[N], s=8, alpha=0.5, color="tab:blue")
ax.axhline(P_B_GIVEN_J_NOTM, color="k", ls="--", lw=1, label=f"exact {P_B_GIVEN_J_NOTM:.3f}")
ax.set_xscale("log"); ax.set_xlabel("number of samples N"); ax.set_ylabel("estimate of P(B | J, ¬M)")
ax.set_title(f"rejection sampling: {reps} repetitions per N"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The step-by-step draw follows the five steps of ancestral sampling: B, then E, then A given
(B, E), then J and M given A. (Our random draw happens to produce the most common world, "nothing happens"; a luckier
draw could give, say, B = 1 from $P(B)$, E = 0 from $P(E)$, A = 1 from $P(A\mid B{=}1,E{=}0)$, J = 1 from
$P(J\mid A{=}1)$ and M = 0 from $P(M\mid A{=}1)$.) With a million samples, 55,527 (5.55 %) satisfy the evidence,
matching $P(J,\neg M)=0.0555$, and $N_B/N_c=0.1122$ is within 0.002 of the exact 0.1138. The scatter shows the price of
approximation: with $N=100$ only about five samples survive the rejection, and the estimates scatter between 0 and 0.5
(standard deviation 0.13); the spread shrinks roughly like $1/\sqrt{N}$ (0.034, 0.014 and 0.004 for $N=10^3,10^4,10^5$)
— a hundred times more samples for ten times less error (the standard error of a proportion, Module 05).

**Rare evidence is the weak spot.** Rejection sampling throws away every sample that disagrees with the evidence. The
rarer the evidence, the more samples are wasted.

In [ ]:
# ── 4.3 The rarer the evidence, the fewer samples survive ──
cases = [{"J": 1}, {"J": 1, "M": 0}, {"J": 1, "M": 0, "E": 1}, {"J": 1, "M": 1, "E": 1, "A": 0}]
sub = samples.iloc[:100_000]
print(f"{'evidence':28s} {'P(evidence)':>11s} {'kept of 10^5':>12s} {'estimate':>9s} {'exact':>8s}")
for ev in cases:
    exact = enumerate_query(ALARM_PARENTS, ALARM_CPTS, "B", ev)
    est, n_c = rejection_estimate(sub, "B", ev)
    label = ", ".join(f"{k}={v}" for k, v in ev.items())
    print(f"{label:28s} {exact[1].sum():11.5f} {n_c:12,d} {est:9.4f} {exact[0][1]:8.4f}")

🔍 **What just happened?** For common evidence ("John called", 14 % of days) rejection sampling works well. For the
rarest evidence set — an earthquake, no alarm, yet both neighbours call (about 4 days in 10,000) — only 38 of 100,000
samples survive, none of them contains a burglary, and the estimate is 0 (exact 0.005). Even the third case (938 samples
kept) is off by 0.005. In real networks the evidence is a full set of measurements and is *always* rare. Smarter
samplers fix this: **likelihood weighting** (fix the evidence and weight each sample by how well it explains it) and
**Gibbs sampling / MCMC** (sample each hidden variable from its distribution given its Markov blanket, Module 16 —
Module 21 develops MCMC).

✏️ **Your turn.** With the 10⁶ samples, estimate $P(B\mid J{=}1,M{=}1)$ by rejection and compare with
`enumerate_query`. How many samples were kept?

### 📘 Lecture notes — Stochastic inference by sampling

- **Probabilities from samples.** $\hat P(B{=}1)=\frac1n\sum_i\mathbf 1\{b_i=1\}$ (e.g. 4 heads in 8 flips → 4/8);
  $\hat P(B{=}1\mid E{=}1)$ = fraction of the samples with $E=1$ that have $B=1$.
- **Generating joint samples.** Sampling from the full $2^n$-entry joint table is bad (exponential). **Ancestral
  sampling** follows the graph: (1) B from $P(B)$; (2) E from $P(E)$; (3) A from $P(A\mid B,E)$ given the drawn B,
  E; (4) J from $P(J\mid A)$; (5) M from $P(M\mid A)$.
- **Conditional queries.** With $N$ samples, $N_c$ = number satisfying the condition ($J,\neg M$), $N_B$ = number
  where the joint event ($B,J,\neg M$) holds:
  $$P(B\mid J,\neg M)=\frac{P(B,J,\neg M)}{P(J,\neg M)}\approx\frac{N_B/N}{N_c/N}=\frac{N_B}{N_c}.$$
- **Accuracy.** The error shrinks like $1/\sqrt{N_c}$; rare evidence leaves few usable samples (rejection
  sampling starves).
- **Summary of inference.** Enumeration: exact, slow. Variable elimination: exact, faster. Stochastic inference:
  inexact, fast.

> 🎓 **Exam tip.** Know why the estimator is $N_B/N_c$ (the $N$'s cancel) and that its precision depends on $N_c$, the number
    of samples consistent with the evidence, not on $N$.

---
## 5 · Learning the CPTs when the graph is known

> 📍 **Where we are.** So far the CPTs were given. What is there to "learn"? A Bayesian network is a probability
> distribution + a graph, so there are two learning problems: learn the probability distribution when the graph is
> known (this section), or learn the distribution *and* the graph (Sections 6–10). This is the course's
> **two-step template** again (Module 05): specify a probability model (the BN), then estimate its parameters by
> maximum likelihood (MLE) or MAP.

The parameters $\theta$ are all CPT entries: $P(B{=}1)$, $P(E{=}1)$, $P(A{=}1\mid B{=}1,E{=}1)$, …, $P(M{=}1\mid A{=}0)$
— ten numbers for the alarm network. The MLE is $\hat\theta=\arg\max_\theta\log P(\text{data}\mid\theta)$. With $N$
fully observed samples (rows $i=1,\dots,N$), the log-likelihood **decomposes**:

$$\log\prod_{i=1}^{N}P(B_i,E_i,A_i,J_i,M_i)=\sum_i\log P(B_i)+\sum_i\log P(E_i)+\sum_i\log P(A_i\mid B_i,E_i)+\sum_i\log P(J_i\mid A_i)+\sum_i\log P(M_i\mid A_i).$$

*In words:* the log of a product is the sum of the logs, and each node contributes its own sum that involves only its
own CPT. So **estimating the Bayesian network's parameters = estimating each conditional probability distribution
separately**.

Each CPT row is a coin (Bernoulli) whose MLE you derived in Module 05. For the row "A given $B{=}b,E{=}e$", let
$N_{1}$ and $N_{0}$ count the samples with that parent configuration and $A=1$ or $A=0$. Maximising
$N_1\log\theta+N_0\log(1-\theta)$ (set the derivative $N_1/\theta-N_0/(1-\theta)$ to zero) gives

$$\hat\theta_{\text{MLE}}=\frac{N_1}{N_1+N_0}=\frac{N(A{=}1,B{=}b,E{=}e)}{N(B{=}b,E{=}e)},\qquad
\hat\theta_{\text{MAP}}=\frac{N_1+\alpha}{N_1+N_0+2\alpha}.$$

*In words:* the MLE is a conditional frequency — among the samples with these parent values, the fraction with $A=1$.
The MAP estimate with a Beta($\alpha{+}1,\alpha{+}1$) prior adds $\alpha$ imaginary samples of each outcome
("pseudo-counts", Laplace smoothing for $\alpha=1$), which keeps estimates away from 0 and 1 and defines them even when a
parent configuration never occurs.

In [ ]:
# ── 5.1 MLE and MAP of the alarm CPTs from sampled data ──
def fit_cpts(df, parents, alpha=0.0):
    """CPTs (P(node = 1 | parent values)) estimated by counting; alpha = pseudo-counts (0 → MLE)."""
    cpts, counts = {}, {}
    for node, pa in parents.items():
        cpts[node], counts[node] = {}, {}
        for pa_vals in itertools.product([0, 1], repeat=len(pa)):
            rows = np.ones(len(df), bool)
            for parent, value in zip(pa, pa_vals):
                rows &= df[parent].to_numpy() == value
            n1, n = int(df.loc[rows, node].sum()), int(rows.sum())
            cpts[node][pa_vals] = (n1 + alpha) / (n + 2 * alpha) if n + 2 * alpha > 0 else np.nan
            counts[node][pa_vals] = n
    return cpts, counts


train = ancestral_sample(ALARM_PARENTS, ALARM_CPTS, 1_000, np.random.default_rng(SEED + 5))
mle, n_rows = fit_cpts(train, ALARM_PARENTS)
mapest, _ = fit_cpts(train, ALARM_PARENTS, alpha=1.0)
rows = [{"CPT entry": f"P({v}=1|{','.join(f'{p}={x}' for p, x in zip(ALARM_PARENTS[v], vals))})",
         "true": ALARM_CPTS[v][vals], "samples with these parents": n_rows[v][vals],
         "MLE": round(mle[v][vals], 3), "MAP (α=1)": round(mapest[v][vals], 3)}
        for v in ALARM_NODES for vals in ALARM_CPTS[v]]
print("learned from N = 1,000 samples:")
print(pd.DataFrame(rows).to_string(index=False))

🔍 **What just happened?** With 1,000 samples most entries are close to the truth, but look at the column "samples with
these parents": rows whose parent configuration is rare are estimated from a handful of samples. $B{=}1,E{=}1$ happens
on 0.5 % of days, so $P(A{=}1\mid B{=}1,E{=}1)$ is estimated from just 10 samples in this run (MLE 0.90, MAP 0.83, truth
0.95); with so few samples the MLE can easily be exactly 0 or 1, and the MAP estimate is pulled towards ½. The precision of each CPT entry depends on how often its parent configuration occurs —
which is why CPTs of nodes with many parents need so much data (Module 15's parameter counting, seen from the data
side).

In [ ]:
# ── 5.2 How fast do the learned CPTs converge? (error vs N, 20 repetitions) ──
rng = np.random.default_rng(SEED + 52)
sizes = [50, 200, 1_000, 5_000, 20_000, 100_000]
entries = [(v, vals) for v in ALARM_NODES for vals in ALARM_CPTS[v]]
err = {k: np.zeros((len(sizes), 20)) for k in ("MLE", "MAP")}
row_err = np.zeros((len(sizes), len(entries)))
for i, N in enumerate(sizes):
    for r in range(20):
        df = ancestral_sample(ALARM_PARENTS, ALARM_CPTS, N, rng)
        for name, a in (("MLE", 0.0), ("MAP", 1.0)):
            est = fit_cpts(df, ALARM_PARENTS, alpha=a)[0]
            e = np.array([abs(np.nan_to_num(est[v][vals], nan=0.5) - ALARM_CPTS[v][vals]) for v, vals in entries])
            err[name][i, r] = e.mean()
            if name == "MLE":
                row_err[i] += e / 20

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for name, col in (("MLE", "tab:blue"), ("MAP", "tab:orange")):
    a1.loglog(sizes, err[name].mean(1), "o-", color=col, label=f"{name}" + (" (α = 1)" if name == "MAP" else ""))
a1.loglog(sizes, 0.5 / np.sqrt(sizes), "k--", lw=1, label=r"$\propto 1/\sqrt{N}$")
a1.set_xlabel("number of training samples N"); a1.set_ylabel("mean |estimate − truth|"); a1.legend()
a1.set_title("average error over the 10 CPT entries")
for j, (v, vals) in enumerate(entries):
    if v == "A":
        a2.loglog(sizes, row_err[:, j], "o-", label=f"P(A|B={vals[0]},E={vals[1]})")
a2.set_xlabel("number of training samples N"); a2.set_ylabel("mean |MLE − truth|")
a2.set_title("rare parent configurations converge last"); a2.legend(fontsize=7)
plt.tight_layout(); plt.show()
for name in ("MLE", "MAP"):
    print(f"mean error {name}: " + "  ".join(f"N={N:,}: {e:.4f}" for N, e in zip(sizes, err[name].mean(1))))
a_rows = [j for j, (v, _) in enumerate(entries) if v == "A"]
print("MLE error of the four P(A|B,E) rows at N = 10^5:",
      {f"B={entries[j][1][0]},E={entries[j][1][1]}": round(float(row_err[-1, j]), 4) for j in a_rows})

🔍 **What just happened?** Both estimators converge at the $1/\sqrt N$ rate of every frequency estimate. The prior
helps only for the smallest sample size (mean error 0.115 for MAP vs 0.128 for MLE at $N=50$), where it prevents wild
0/1 values from tiny counts (rows never observed are scored as ½ for the MLE here, an arbitrary but generous choice).
For $N$ between 200 and 20,000 the MAP estimate is slightly *worse*: its pull towards ½ biases the many entries that are
close to 0 or 1 (0.05, 0.95). For large $N$ the prior's influence vanishes and the two coincide. The right panel shows the
four rows of $P(A\mid B,E)$: at $N=10^5$ the row $B{=}1,E{=}1$ is still about 13 times less accurate than the row
$B{=}0,E{=}0$ — matching it would need $0.855/0.005\approx170$ times more data, the ratio of how often the two parent
configurations occur.

✏️ **Your turn.** Repeat 5.1 with `alpha=5`. Which entries does the stronger prior improve, and which does it spoil?

### 📘 Lecture notes — Learning the CPTs of a known graph

- **What to learn.** A BN = probability distribution + graph. Two problems: (1) learn the distribution with a known
  graph; (2) learn the distribution and the graph.
- **MLE.** $\theta\leftarrow\arg\max_\theta\log P(\text{data}\mid\theta)$, with
  $\theta=\{P(B{=}1),P(E{=}1),P(A{=}1\mid B{=}1,E{=}1),\dots,P(M{=}1\mid A{=}0)\}$.
- **Decomposition.** $$\log\prod_i P(B_i)P(E_i)P(A_i\mid B_i,E_i)P(J_i\mid A_i)P(M_i\mid A_i)=\sum_i\log P(B_i)+\dots+\sum_i\log P(M_i\mid A_i).$$
  *In words:* five separate sums, one per CPD — estimating the network = estimating each CPD separately.
- **Each CPT entry** is a Bernoulli MLE, a conditional frequency:
  $\hat P(A{=}1\mid B{=}b,E{=}e)=N(A{=}1,B{=}b,E{=}e)/N(B{=}b,E{=}e)$. **MAP** with pseudo-counts:
  $(N_1+\alpha)/(N+2\alpha)$.
- **Data needed.** Each entry's precision depends on how often its parent configuration occurs: rows with rare
  parents (here $B{=}1,E{=}1$, 0.5 % of days) converge last.

> 🎓 **Exam tip.** Be ready to write the MLE of any CPT entry as a ratio of counts, e.g.
    $\hat P(J{=}1\mid A{=}0)=N(J{=}1,A{=}0)/N(A{=}0)$, and to explain why decomposition makes BN learning easy
    *when the graph is known and the data are fully observed* (with hidden variables you need EM, Module 13).

---
## 6 · Learning the graph: why is it hard?

> 📍 **Where we are.** Now the graph itself is unknown. Learning a Bayesian network structure is an open problem in
> general. Two routes: restrict to **trees** (Chow–Liu: exact and fast, Section 7) or search over
> **general DAGs** with a score (heuristic search, Section 9; exact dynamic programming exists but takes exponential
> time).

How many DAGs are there on $n$ labelled nodes? R. W. Robinson's recursion counts them:

$$a(n)=\sum_{k=1}^{n}(-1)^{k+1}\binom{n}{k}\,2^{k(n-k)}\,a(n-k),\qquad a(0)=1 .$$

*In words:* choose a set of $k$ nodes with no parents (the "sources"), let each of them point or not point to each of
the $n-k$ other nodes ($2^{k(n-k)}$ ways), and complete the remaining nodes as any DAG; the alternating signs correct for
over-counting graphs that have more than $k$ sources (inclusion–exclusion). $\binom nk$ ("n choose k") is the number of
ways to choose $k$ items from $n$. For comparison, the number of (undirected) **trees** on $n$ labelled nodes is
$n^{n-2}$ (Cayley's formula).

In [ ]:
# ── 6.1 The number of DAGs explodes super-exponentially ──
n_dags = {0: 1}
for n in range(1, 12):
    n_dags[n] = sum((-1) ** (k + 1) * comb(n, k) * 2 ** (k * (n - k)) * n_dags[n - k] for k in range(1, n + 1))
print(f"{'n':>3} {'DAGs a(n)':>32} {'trees n^(n−2)':>16}")
for n in range(1, 12):
    print(f"{n:>3} {n_dags[n]:>32,} {n ** (n - 2) if n > 1 else 1:>16,}")

🔍 **What just happened?** 3 DAGs on 2 nodes, 25 on 3, 543 on 4, 29,281 on 5 — and about $3.2\times10^{22}$ on the 11
Sachs proteins. Even scoring a billion graphs per second, an exhaustive search would take a million years. Trees are far
fewer ($11^9\approx2.4\times10^{9}$), and — remarkably — the best tree can be found without looking at any of them one by
one.

✏️ **Your turn.** Verify $a(3)=25$ by hand: count the DAGs on three labelled nodes with 0, 1, 2 and 3 edges. (Hint:
with 3 edges, 2 of the 8 orientations are cycles.)

---
## 7 · Chow–Liu trees

> 📍 **Where we are.** Restrict the search to tree-structured networks (every node has at most one parent). Chow and
> Liu (1968) showed that the best such tree can be found exactly with a maximum spanning tree. Trees are a restrictive
> model structure, but they come with efficient learning and inference algorithms.

🧮 **Math refresher — entropy, KL divergence, mutual information** (logs in base 2, so the unit is bits).
- **Entropy** $H(X)=-\sum_x P(x)\log_2P(x)$: the average surprise of $X$ (Module 10 used it for decision trees).
- **Kullback–Leibler divergence** between the true distribution $P(\mathbf X)$, $\mathbf X=(X_1,\dots,X_n)$, and a
  tree-structured network $T(\mathbf X)$ — the quantity Chow–Liu minimises:

$$\mathrm{KL}(P\,\|\,T)=\sum_{\mathbf k}P(\mathbf X=\mathbf k)\,\log_2\frac{P(\mathbf X=\mathbf k)}{T(\mathbf X=\mathbf k)} .$$

*In words:* the average, over configurations drawn from the truth, of how much more probable the truth finds them
than the approximation does. It is $\ge0$, and $=0$ only if $T=P$.
- **Mutual information** of two variables $A$ and $B$ (the weight of a potential edge between them):

$$I(A,B)=\sum_a\sum_b P(a,b)\,\log_2\frac{P(a,b)}{P(a)\,P(b)} .$$

*In words:* how far the pair is from independence — how many bits knowing $A$ tells you about $B$ (Module 16's
conditional mutual information with an empty conditioning set).

**The key result.** For a tree $T$ whose CPDs are the true conditionals $P(x_i\mid x_{\mathrm{pa}(i)})$,

$$\mathrm{KL}(P\,\|\,T)=-H(P)+\sum_{i}H(X_i)-\sum_{(i,j)\in\text{tree}}I(X_i,X_j).$$

*In words:* the first two terms do not depend on the tree at all, so **minimising the KL divergence is the same as
maximising the sum of the mutual informations over the tree's edges.** (Derivation: $\log T(\mathbf x)=\sum_i\log
P(x_i\mid x_{\mathrm{pa}(i)})=\sum_i\big[\log\frac{P(x_i,x_{\mathrm{pa}(i)})}{P(x_i)P(x_{\mathrm{pa}(i)})}+\log P(x_i)\big]$;
average over $P$: each edge contributes $I(X_i,X_{\mathrm{pa}(i)})$ and each node $-H(X_i)$.) Note that the direction
of the edges plays no role. Let us verify the key result on the alarm network by brute force over **all 125** spanning
trees of its five nodes.

In [ ]:
# ── 7.1 Exact entropies and mutual informations of the alarm distribution ──
p = np.array([joint_prob(dict(zip(ALARM_NODES, s)), ALARM_PARENTS, ALARM_CPTS)
              for s in itertools.product([0, 1], repeat=5)]).reshape([2] * 5)       # p[b, e, a, j, m]
ax_of = {v: i for i, v in enumerate(ALARM_NODES)}


def marg(*vs):
    """Exact marginal distribution of the listed alarm variables (axes in the listed order)."""
    out = p.sum(axis=tuple(i for i in range(5) if ALARM_NODES[i] not in vs))
    present = [v for v in ALARM_NODES if v in vs]
    return np.transpose(out, [present.index(v) for v in vs])


def entropy(q):
    q = q[q > 0]
    return float(-np.sum(q * np.log2(q)))


MI = {(u, v): entropy(marg(u)) + entropy(marg(v)) - entropy(marg(u, v)) for u, v in itertools.combinations(ALARM_NODES, 2)}

In [ ]:
# ── 7.2 Brute-force check of the key result: KL divergence of all 125 spanning trees ──
def tree_kl(tree_edges, root="B"):
    """KL(P || T) for the tree with the true pairwise conditionals, oriented away from `root`."""
    logT, done, frontier = np.zeros_like(p), {root}, [root]
    logT += np.log2(marg(root)).reshape([2 if v == root else 1 for v in ALARM_NODES])
    while frontier:
        u = frontier.pop()
        for a, b in tree_edges:
            for parent, child in ((a, b), (b, a)):
                if parent == u and child not in done:
                    cond = marg(parent, child) / marg(parent)[:, None]            # P(child | parent)
                    order = sorted([parent, child], key=ax_of.get)
                    c = cond if order == [parent, child] else cond.T
                    logT += np.log2(c).reshape([2 if v in (parent, child) else 1 for v in ALARM_NODES])
                    done.add(child); frontier.append(child)
    return float(np.sum(p * (np.log2(p) - logT)))


trees = [t for t in itertools.combinations(list(MI), 4) if is_polytree(ALARM_NODES, list(t))]
kl = np.array([tree_kl(t) for t in trees])
mi_sum = np.array([sum(MI[e] for e in t) for t in trees])
const = -entropy(p.ravel()) + sum(entropy(marg(v)) for v in ALARM_NODES)
best = trees[int(np.argmin(kl))]
cmi_BE_given_A = entropy(marg("A", "B")) + entropy(marg("A", "E")) - entropy(marg("A")) - entropy(marg("A", "B", "E"))
print(f"{len(trees)} spanning trees; max |KL − (−H(P) + Σ H(X_i) − Σ I)| = {np.max(np.abs(kl - (const - mi_sum))):.1e} bits")
print(f"best tree: {best}  KL = {kl.min():.4f} bits  (it is the tree with the largest Σ I: {best == trees[int(np.argmax(mi_sum))]})")
print(f"for comparison, I(B;E | A) = {cmi_BE_given_A:.4f} bits (the explaining-away information of Module 16)")
print("pairwise mutual information (bits):", {f"{u}{v}": round(m, 4) for (u, v), m in MI.items()})

In [ ]:
# ── 7.3 Plot: KL divergence vs the sum of mutual informations over the tree's edges ──
fig, ax = plt.subplots(figsize=(5, 3.6))
ax.scatter(mi_sum, kl, s=14, color="tab:blue", label="the 125 spanning trees")
ax.scatter(mi_sum.max(), kl.min(), s=60, color="tab:red", zorder=3, label="Chow–Liu tree (max Σ I)")
xx = np.linspace(mi_sum.min(), mi_sum.max(), 10)
ax.plot(xx, const - xx, "k--", lw=1, label="KL = const − Σ I (slope −1)")
ax.set_xlabel("Σ mutual information over tree edges (bits)"); ax.set_ylabel("KL(P || T) (bits)")
ax.set_title("the key result of Chow & Liu, checked exactly"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** All 125 trees lie *exactly* on a line of slope −1 (deviation at round-off level): the tree
with the largest sum of mutual informations has the smallest KL divergence. That best tree is the alarm network's own
skeleton (B–A, E–A, A–J, A–M) — but its KL divergence is **not zero**. A tree gives every node at most one parent, so it
cannot represent the collider $B\to A\leftarrow E$ (A needs both parents at once). Its KL divergence, 0.0172 bits, is
exactly the conditional mutual information $I(B;E\mid A)$ (printed for comparison; also computed in Module 16) — the information carried by explaining
away, which no tree can capture. Chow–Liu finds the best *tree*; when the truth is not a tree, the best tree is an
approximation.

### 7.1 Kruskal's maximum spanning tree

The Chow–Liu algorithm: (1) for each pair of variables $A,B$, estimate $P(A,B)$, $P(A)$, $P(B)$ from the data; (2)
compute $I(A,B)$ for each pair; (3) find the **maximum spanning tree** with edge weights $I$ (for $N$ variables and all
$N(N-1)/2$ candidate edges this takes polynomial time: $O(N^2\log N)$ with Kruskal's algorithm below, $O(N^2)$ with
Prim's algorithm); (4) pick any root and direct edges away from it; (5) learn the CPDs for this graph. Step 3
uses **Kruskal's algorithm**: start with the empty graph and add edges one by one, each time choosing the edge that is
not yet in the graph, does not introduce a cycle, and has the maximum weight. To check "creates a cycle?" quickly we keep
track of which nodes are already connected (**union–find**: each node points towards a representative of its group).
We run it on a classic seven-node worked example (after A. Singh and C. Guestrin), with mutual informations between
1/5 and 1/15 bits on eleven candidate edges.

In [ ]:
# ── 7.4 Kruskal's algorithm on the seven-node worked example ──
MST_EXAMPLE = {("A", "B"): Fraction(1, 7), ("A", "D"): Fraction(1, 5), ("B", "C"): Fraction(1, 8),
            ("B", "D"): Fraction(1, 9), ("B", "E"): Fraction(1, 7), ("C", "E"): Fraction(1, 5),
            ("D", "E"): Fraction(1, 15), ("D", "F"): Fraction(1, 6), ("E", "F"): Fraction(1, 8),
            ("E", "G"): Fraction(1, 9), ("F", "G"): Fraction(1, 11)}


def kruskal_max(nodes, weights, verbose=False):
    """Maximum spanning tree: add edges from heaviest to lightest unless they close a cycle (union–find)."""
    group = {v: v for v in nodes}

    def find(v):
        while group[v] != v:
            v = group[v]
        return v

    tree = []
    for (u, v), w in sorted(weights.items(), key=lambda kv: -kv[1]):     # stable: ties keep listing order
        ru, rv = find(u), find(v)
        accept = ru != rv
        if accept:
            group[ru] = rv
            tree.append((u, v))
        if verbose:
            print(f"  {u}–{v} (weight {w}): {'add' if accept else 'reject — would close a cycle'}")
    return tree


cl_tree = kruskal_max(list("ABCDEFG"), MST_EXAMPLE, verbose=True)
print("maximum spanning tree:", cl_tree, "| total weight =", sum(MST_EXAMPLE[e] for e in cl_tree),
      f"≈ {float(sum(MST_EXAMPLE[e] for e in cl_tree)):.4f}")

In [ ]:
# ── 7.5 Direct the tree away from a root (step 4) and draw it next to the weighted graph ──
def orient_tree(tree_edges, root):
    """Directed edges (parent, child) pointing away from `root` (breadth-first search)."""
    directed, seen, queue = [], {root}, [root]
    while queue:
        u = queue.pop(0)
        for a, b in tree_edges:
            for x, y in ((a, b), (b, a)):
                if x == u and y not in seen:
                    directed.append((x, y)); seen.add(y); queue.append(y)
    return directed


POS_MST = {"A": (0.1, 0.85), "B": (0.38, 0.68), "C": (0.65, 0.85), "D": (0.02, 0.45), "E": (0.62, 0.45),
           "F": (0.22, 0.08), "G": (0.8, 0.08)}
directed_mst = orient_tree(cl_tree, "A")
print("directed from root A:", directed_mst)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
draw_graph(list(MST_EXAMPLE), POS_MST, directed=False, ax=axes[0],
           edge_labels=[str(w) for w in MST_EXAMPLE.values()], title="mutual-information weights")
draw_graph(cl_tree, POS_MST, directed=False, ax=axes[1], title="Kruskal: maximum spanning tree")
for u, v in cl_tree:                                            # tree edges highlighted in green
    (x0, y0), (x1, y1) = POS_MST[u], POS_MST[v]
    axes[1].plot([x0, x1], [y0, y1], color="tab:green", lw=6, alpha=0.35, zorder=0)
draw_graph(directed_mst, POS_MST, ax=axes[2], highlight=["A"], title="rooted at A: a directed tree")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Kruskal adds A–D and C–E (1/5 each), D–F (1/6), A–B and B–E (1/7 each), rejects B–C and E–F
(1/8; each would close a cycle) and B–D (1/9), adds E–G (1/9), and rejects F–G and D–E. The tree A–B, A–D, B–E, C–E, D–F,
E–G (total weight ≈ 0.96) is the maximum spanning tree (green, middle), and rooting it at A gives the arrows A→B, A→D,
B→E, D→F, E→C, E→G (right). Ties (two edges of weight 1/5, 1/7, 1/8 and 1/9) do not matter here: whichever order they
are processed in, the same tree results. (Hand animations of this example sometimes reject B–D right after A–B is
added; strict Kruskal order considers B–E (1/7) and B–C (1/8) first — the outcome is the same.)

Now the full Chow–Liu pipeline on **data** sampled from the alarm network: estimate the mutual informations from
counts, build the tree, orient it, fit the CPDs. The functions are written for variables with any number of levels, so
that we can reuse them on the Sachs proteins.

In [ ]:
# ── 7.6 Chow–Liu from data (any number of levels per variable) ──
def mutual_info_matrix(X, card):
    """Plug-in mutual information (bits) between all pairs of integer-coded columns of X."""
    J = X.shape[1]
    M = np.zeros((J, J))
    for i, j in itertools.combinations(range(J), 2):
        joint = np.bincount(X[:, i] * card[j] + X[:, j], minlength=card[i] * card[j]).reshape(card[i], card[j])
        pij = joint / joint.sum()
        pi, pj = pij.sum(1, keepdims=True), pij.sum(0, keepdims=True)
        keep = pij > 0
        M[i, j] = M[j, i] = np.sum(pij[keep] * np.log2(pij[keep] / (pi @ pj)[keep]))
    return M


def chow_liu(X, names, card, root):
    """Chow–Liu tree: mutual informations → maximum spanning tree → edges directed away from `root`."""
    M = mutual_info_matrix(X, card)
    weights = {(names[i], names[j]): M[i, j] for i, j in itertools.combinations(range(len(names)), 2)}
    return orient_tree(kruskal_max(names, weights), root), M


alarm_data = ancestral_sample(ALARM_PARENTS, ALARM_CPTS, 5_000, np.random.default_rng(SEED + 75))
cl_alarm, M_alarm = chow_liu(alarm_data.to_numpy(), ALARM_NODES, [2] * 5, root="B")
print("Chow–Liu tree from 5,000 samples:", cl_alarm)
true_skeleton = {frozenset(e) for e in alarm_edges}
print("same skeleton as the true network:", {frozenset(e) for e in cl_alarm} == true_skeleton)

🔍 **What just happened?** From 5,000 simulated days, Chow–Liu recovers the alarm network's skeleton exactly. Rooted
at B, it orients the edges B→A, A→E, A→J, A→M: the arrow between A and E points the "wrong" way, because a rooted tree
can give A only one parent. Remember this: tree learning gets the *connections* right here, but the *directions* come
from the arbitrary root, not from the data (Section 10).

✏️ **Your turn.** Re-run with 200 samples instead of 5,000 (change the `ancestral_sample` call). Is the skeleton still
recovered? Which edge is the first to go wrong, and why? (Look at `M_alarm`: which true edge has the weakest mutual
information?)

### 📘 Lecture notes — Structure learning and Chow–Liu trees

- **Structure learning is hard.** Learning the graph is an open problem in general: $3.2\times10^{22}$ DAGs on 11
  nodes. Options: restrict to **trees** (each child has one parent; restrictive, but efficient learning and
  inference — Chow–Liu), or search **general DAGs** (heuristic search, efficient but inexact; exact dynamic
  programming, exponential time).
- **Chow–Liu goal.** Find the tree-structured $T(\mathbf X)$ closest to the true $P(\mathbf X)$:
  $$\mathrm{KL}(P\|T)=\sum_{\mathbf k}P(\mathbf X{=}\mathbf k)\log\frac{P(\mathbf X{=}\mathbf k)}{T(\mathbf X{=}\mathbf k)}.$$
- **Key result.** $\mathrm{KL}(P\|T)=-H(P)+\sum_iH(X_i)-\sum_{\text{edges}}I(X_i,X_j)$, so minimising KL = finding the
  tree that **maximises the sum of mutual informations** $I(A,B)=\sum_{a,b}P(a,b)\log\frac{P(a,b)}{P(a)P(b)}$ over
  its edges.
- **Algorithm.** (1) Estimate $P(A,B)$, $P(A)$, $P(B)$ for every pair; (2) compute $I(A,B)$; (3) maximum spanning
  tree with weights $I$ (polynomial time: $O(N^2\log N)$ with Kruskal, $O(N^2)$ with Prim); (4) pick a root,
  direct edges outward; (5) learn the CPDs.
- **Kruskal's algorithm.** Start empty; repeatedly add the heaviest edge that is not yet in the graph and does not
  create a cycle. Worked example (A–G, weights 1/5 … 1/15): tree A–B, A–D, B–E, C–E, D–F, E–G.
- **Limits.** A tree cannot represent a collider (KL of the best alarm tree = $I(B;E\mid A)$ = 0.0172 bits > 0).

> 🎓 **Exam tip.** Kruskal adds the **heaviest** edge that does not close a cycle (maximum, not minimum spanning tree); with
    n nodes you stop after n − 1 edges. The edge directions are set by the arbitrary root and do not change the
    likelihood.

---
## 8 · Signalling data: a Chow–Liu tree of (simulated) T-cell signalling

> 📍 **Where we are.** Everything is in place to learn from single-cell measurements. Sachs *et al.* (2005) measured
> 11 signalling molecules by flow cytometry in thousands of T cells, discretised each into low/medium/high, and learned
> a Bayesian network. We use **simulated** cells of the same design: 1,800 unperturbed cells, each protein cut at its
> tertiles, generated from the 20-edge consensus network — the **true graph** against which we grade what we learn.

In [ ]:
# ── 8.1 SIMULATED single-cell signalling data: 11 molecules, discretised into low/medium/high ──
sim = simulate_signaling_cytometry("discrete")              # unperturbed cells only (default)
data = sim["data"]
print(data.shape, "|", sim["source"])
X_sachs = data[SACHS_PROTEINS].to_numpy()
card_sachs = [3] * len(SACHS_PROTEINS)                         # levels 0/1/2 = low/medium/high
print("cells per level:", {p: np.bincount(data[p]).tolist() for p in ["raf", "pka", "pkc"]}, "…")
print(data.head().to_string())
true_edges = sim["edges"]                                       # = SACHS_CONSENSUS_EDGES, the generating graph
all_cells = simulate_signaling_cytometry("discrete", observational_only=False)
print("full design:", len(all_cells["data"]), "cells; perturbation targets:",
      sorted(set(all_cells["intervention"]) - {"none"}))

🔍 **What just happened?** The simulated data have 1,800 unperturbed cells × 11 proteins; because each protein was cut
at its tertiles, every level holds exactly 600 cells. The full design also contains cells in which one protein was
inhibited or activated (as in the original study: PKC, PKA, Akt, PIP2, Mek); we keep those for the discussion in
Section 10 and learn only from the unperturbed cells. `true_edges` is the graph the data were generated from.

In [ ]:
# ── 8.2 Mutual information between all pairs of proteins ──
names = SACHS_PROTEINS
M_sachs = mutual_info_matrix(X_sachs, card_sachs)
true_skeleton = {frozenset(e) for e in true_edges}
pairs = sorted(itertools.combinations(range(11), 2), key=lambda ij: -M_sachs[ij])
print("the 12 most dependent pairs (bits); ✓ = joined by an edge in the true network")
for i, j in pairs[:12]:
    print(f"  {names[i]:5s}–{names[j]:5s} {M_sachs[i, j]:.4f} {'✓' if frozenset((names[i], names[j])) in true_skeleton else ''}")
mi_edges = [M_sachs[names.index(a), names.index(b)] for a, b in true_edges]
mi_non = [M_sachs[i, j] for i, j in itertools.combinations(range(11), 2) if frozenset((names[i], names[j])) not in true_skeleton]
print(f"🧪 MI over the 20 true edges: median {np.median(mi_edges):.4f}, smallest {min(mi_edges):.4f} bits "
      f"| over the 35 non-edges: median {np.median(mi_non):.4f}, largest {max(mi_non):.4f} bits")
weak = sorted(zip(mi_edges, true_edges))[:3]
print("   the three weakest true edges:", ", ".join(f"{a}→{b} ({m:.4f})" for m, (a, b) in weak))

fig, ax = plt.subplots(figsize=(4.8, 4.2))
im = ax.imshow(M_sachs, cmap="viridis")
ax.set_xticks(range(11)); ax.set_xticklabels(names, rotation=90, fontsize=7)
ax.set_yticks(range(11)); ax.set_yticklabels(names, fontsize=7)
for a, b in true_edges:
    i, j = names.index(a), names.index(b)
    ax.scatter([j, i], [i, j], marker="s", s=45, facecolors="none", edgecolors="white", lw=1)
ax.set_title("mutual information (bits), SIMULATED cells\nwhite squares: true edges"); fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The most dependent pairs are almost all true edges — Mek–Erk (0.73 bits), Erk–PKA, Mek–PKA,
Raf–Mek, PKA–p38, … — with one exception in the top 12: Raf–Erk (0.46 bits), which is *not* an edge but is linked by
the chain Raf → Mek → Erk and the common parent PKA (the same pair Module 16 tested). 🧪 Because we know the truth we
can grade the whole matrix: the median MI is 0.45 bits over the true edges and 0.03 bits over the non-edges, yet the
ranges overlap — the strongest non-edge beats ten of the true edges. The lipid module (PIP3, PLCγ, PIP2) is tied to
the rest of the network only by the three weakest true edges: PIP3 → Akt (0.041 bits), PLCγ → PKC (0.044) and
PIP2 → PKC, which carries just 0.0005 bits. As in Modules 15–16, parallel routes of opposite sign nearly cancel (PIP2 is consumed when PLCγ is active); the
discrete version of the simulator draws its own edge weights, so here the cancelled edge is PIP2 → PKC. Dependence that
is not expressed in the data cannot be learned from it. (With real cells this is the first thing to check: how much
dependence do the unperturbed measurements actually carry along the edges you hope to learn? In a steady state where
much of the pathway barely varies, it can be far less than in these simulated cells.)

In [ ]:
# ── 8.3 The Chow–Liu tree of the simulated data, graded against the true network (🧪) ──
SACHS_POS = {"plc": (0.08, 0.88), "pip3": (0.42, 0.97), "pip2": (0.08, 0.5), "pkc": (0.4, 0.68),
             "pka": (0.62, 0.42), "raf": (0.15, 0.3), "mek": (0.3, 0.05), "erk": (0.65, 0.05),
             "akt": (0.95, 0.2), "p38": (0.8, 0.8), "jnk": (0.97, 0.55)}
all_pairs = [frozenset(pr) for pr in itertools.combinations(names, 2)]


def skeleton_scores(edges):
    """Precision, recall, F1 of an edge list's skeleton against the true skeleton (55 possible pairs)."""
    learned = {frozenset(e) for e in edges}
    return precision_recall_f1([pr in true_skeleton for pr in all_pairs], [pr in learned for pr in all_pairs])


def draw_with_matches(ax, edges, title, directed=True):
    """Draw a learned network; edges present in the true skeleton get a green underlay."""
    for u, v in edges:
        if frozenset((u, v)) in true_skeleton:
            (x0, y0), (x1, y1) = SACHS_POS[u], SACHS_POS[v]
            ax.plot([x0, x1], [y0, y1], color="tab:green", lw=7, alpha=0.35, solid_capstyle="round", zorder=0)
    draw_graph(edges, SACHS_POS, directed=directed, ax=ax, radius=0.055, title=title)


cl_sachs, _ = chow_liu(X_sachs, names, card_sachs, root="pip3")
prec, rec, f1 = skeleton_scores(cl_sachs)
print("Chow–Liu tree (rooted at pip3):", cl_sachs)
print(f"🧪 skeleton vs the true graph: precision {prec:.2f}, recall {rec:.2f}, F1 {f1:.2f}")
print("   learned edges not in the true graph:", [e for e in cl_sachs if frozenset(e) not in true_skeleton])
mi_of = lambda a, b: M_sachs[names.index(a), names.index(b)]
print(f"   MI of the bridge PLCγ–Akt: {mi_of('plc', 'akt'):.3f} bits vs the true bridges pip3–akt {mi_of('pip3', 'akt'):.3f}, "
      f"plc–pkc {mi_of('plc', 'pkc'):.3f}, pip2–pkc {mi_of('pip2', 'pkc'):.4f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
draw_with_matches(axes[0], cl_sachs, f"Chow–Liu tree, SIMULATED cells\n(green = true edge; precision {prec:.2f})")
draw_graph(true_edges, SACHS_POS, ax=axes[1], radius=0.055, title="true network (20 edges)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** 🧪 The tree has 10 edges (always $n-1$), and 9 of them are true edges: **precision 0.90**.
Recall is 0.45, but a tree can contain at most 10 of the 20 true edges, so 0.5 was the best possible. The single wrong
edge, PLCγ–Akt, is the tree's bridge between the lipid module and the proteins: the true bridges (PIP3 → Akt, 0.041
bits; PLCγ → PKC, 0.044; PIP2 → PKC, 0.0005) are all weaker than the indirect PLCγ–Akt dependence (0.058 bits, through
PIP3 and through PLCγ → PKC → PKA → Akt), and a tree must connect the two modules somehow. Remember also that the
arrows were set by the root (PIP3), not learned. A tree is a strong restriction for a signalling network in which PKA
and PKC each regulate five or six proteins; the next section lifts it.

✏️ **Your turn.** Choose another root (e.g. `root="pkc"`). Which arrows change? Does precision change?

---
## 9 · Score-based search: hill climbing with BIC

> 📍 **Where we are.** For general DAGs the naïve approach — compute the score of every structure and pick the best —
> is impossible: the search space is exponentially large (Section 6), and keeping every candidate acyclic is itself
> a challenge. The standard heuristic is local search guided by a **score**.

**Which score?** The maximised log-likelihood alone always prefers *more* edges: adding a parent can never lower the
fit, so it would return the complete DAG (Module 15: 31 parameters for five binary variables). We need a penalty for
complexity — the **Bayesian information criterion** (Schwarz 1978; Module 23 discusses model selection in general):

$$\mathrm{BIC}(G)=\sum_{i=1}^{n}\Big[\underbrace{\sum_{j}\sum_{k}N_{ijk}\log\frac{N_{ijk}}{N_{ij}}}_{\text{max. log-likelihood of node } i}-\frac{(r_i-1)\,q_i}{2}\log N\Big].$$

*In words:* for each node $i$, $N_{ijk}$ counts the cells in which its parents are in configuration $j$ and the node
has value $k$ ($N_{ij}=\sum_k N_{ijk}$); the first term is the log-likelihood at the MLE (Section 5); the second subtracts
$\tfrac12\log N$ for each of the node's $(r_i-1)q_i$ free parameters ($r_i$ levels, $q_i$ parent configurations). Higher is
better. With $N=1{,}800$ cells, each extra parameter must raise the log-likelihood by $\tfrac12\log1800\approx3.7$ to pay
for itself.

Crucially, BIC is a **sum over nodes** of a term that depends only on that node and its parents ("decomposable").
Changing one edge changes at most two terms, so each candidate move is cheap to evaluate.

**Hill climbing.** Start from some DAG (e.g. the empty graph). Repeatedly consider every graph that
differs by one **add**, **delete** or **reverse** of an edge (adds and reversals that would create a directed cycle are
not allowed), move to the best-scoring one, and stop when no move improves the score; return the best network. The
figure after the next cell shows the three search operators on a four-node example.

In [ ]:
# ── 9.1 Decomposable BIC with a cache of family scores ──
N_CELLS = len(X_sachs)
_family_cache = {}


def family_bic(child, parents):
    """BIC term of one node given a tuple of parent indices (Sachs data), cached."""
    key = (child, parents)
    if key not in _family_cache:
        cfg = np.zeros(N_CELLS, int)
        for p in parents:                                  # mixed-radix code of the parent configuration
            cfg = cfg * card_sachs[p] + X_sachs[:, p]
        q, r = int(np.prod([card_sachs[p] for p in parents])), card_sachs[child]
        N_jk = np.bincount(cfg * r + X_sachs[:, child], minlength=q * r).reshape(q, r).astype(float)
        N_j = N_jk.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            loglik = np.where(N_jk > 0, N_jk * np.log(N_jk / N_j), 0.0).sum()
        _family_cache[key] = loglik - 0.5 * np.log(N_CELLS) * q * (r - 1)
    return _family_cache[key]


def bic_score(parent_sets):
    """BIC of a DAG given as a list of parent-index sets (one per protein)."""
    return sum(family_bic(i, tuple(sorted(pa))) for i, pa in enumerate(parent_sets))


def to_parent_sets(edges):
    """Edge list (names) → list of parent-index sets."""
    sets = [set() for _ in names]
    for a, b in edges:
        sets[names.index(b)].add(names.index(a))
    return sets


for label, edges in [("empty graph", []), ("Chow–Liu tree", cl_sachs), ("true network", true_edges)]:
    print(f"BIC of the {label:18s}: {bic_score(to_parent_sets(edges)):10.1f}  ({len(edges)} edges)")

🔍 **What just happened?** The Chow–Liu tree scores far better than the empty graph, and the true network better
still (−16,514.7 vs −16,689.0): with these informative simulated data its 20 edges pay for their parameters, even though
some of them (a three-level protein with three three-level parents has 54 free numbers) cost a lot. With weaker
dependences — fewer cells, or a pathway that barely varies — the same true network could score *worse* than a sparse
tree: the score judges what the data at hand support, not what is biologically true.

In [ ]:
# ── 9.2 Would a new arrow create a directed cycle? ──
def reaches(parent_sets, start, target):
    """True if `target` can be reached from `start` by following arrows (parent → child)."""
    children = [[c for c in range(len(parent_sets)) if v in parent_sets[c]] for v in range(len(parent_sets))]
    stack, seen = [start], {start}
    while stack:
        u = stack.pop()
        if u == target:
            return True
        for c in children[u]:
            if c not in seen:
                seen.add(c); stack.append(c)
    return False


# the three search operators on a four-node DAG (A → C ← B, C → D)
start = [("A", "C"), ("B", "C"), ("C", "D")]
pos4 = {"A": (0.15, 0.85), "B": (0.85, 0.85), "C": (0.5, 0.5), "D": (0.5, 0.1)}
moves = [("current DAG", start, True), ("delete B→C", [("A", "C"), ("C", "D")], True),
         ("add B→D", start + [("B", "D")], True), ("reverse B→C", [("A", "C"), ("C", "B"), ("C", "D")], True),
         ("add D→B: illegal (cycle B→C→D→B)", start + [("D", "B")], False)]
fig, axes = plt.subplots(1, 5, figsize=(12, 2.8))
for ax, (title, edges, legal) in zip(axes, moves):
    draw_graph(edges, pos4, ax=ax, radius=0.09, highlight=[] if legal else ["B", "C", "D"])
    ax.set_title(title, fontsize=8, color="0.2" if legal else "tab:red")
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.3 Hill climbing over DAGs: add, delete, reverse ──
def hill_climb(start, max_parents=4, verbose=False):
    """Greedy local search on BIC; returns (parent sets, score history)."""
    G = [set(pa) for pa in start]
    history = [bic_score(G)]
    while True:
        best_gain, best_move = 1e-9, None
        for i, j in itertools.permutations(range(len(G)), 2):
            fam_j = family_bic(j, tuple(sorted(G[j])))
            if i in G[j]:                                                    # edge i → j exists
                gain = family_bic(j, tuple(sorted(G[j] - {i}))) - fam_j       # delete it
                if gain > best_gain:
                    best_gain, best_move = gain, ("delete", i, j)
                G[j].discard(i)                                              # reverse it: j → i
                if len(G[i]) < max_parents and not reaches(G, i, j):
                    gain = (family_bic(j, tuple(sorted(G[j]))) - fam_j
                            + family_bic(i, tuple(sorted(G[i] | {j}))) - family_bic(i, tuple(sorted(G[i]))))
                    if gain > best_gain:
                        best_gain, best_move = gain, ("reverse", i, j)
                G[j].add(i)
            elif i not in G[j] and j not in G[i] and len(G[j]) < max_parents and not reaches(G, j, i):
                gain = family_bic(j, tuple(sorted(G[j] | {i}))) - fam_j      # add i → j (no cycle)
                if gain > best_gain:
                    best_gain, best_move = gain, ("add", i, j)
        if best_move is None:
            return G, history                                                # no move improves the score
        op, i, j = best_move
        if op in ("delete", "reverse"):
            G[j].discard(i)
        if op == "reverse":
            G[i].add(j)
        if op == "add":
            G[j].add(i)
        history.append(history[-1] + best_gain)
        if verbose:
            print(f"  {op:7s} {names[i]:>4s} → {names[j]:<4s}  BIC gain {best_gain:8.1f}  → {history[-1]:9.1f}")


G_hc, hist_hc = hill_climb([set() for _ in names], verbose=True)
hc_edges = [(names[p], names[c]) for c in range(len(names)) for p in sorted(G_hc[c])]
print(f"local maximum after {len(hist_hc) - 1} moves: BIC {hist_hc[-1]:.1f}, {len(hc_edges)} edges")

ix = names.index
gain = lambda child, new, old=(): family_bic(ix(child), tuple(sorted(ix(p) for p in old + (new,)))) - \
    family_bic(ix(child), tuple(sorted(ix(p) for p in old)))
print(f"\nmove 1 (empty graph): mek → erk gains {gain('erk', 'mek'):.2f}, erk → mek gains {gain('mek', 'erk'):.2f}  (a tie)")
print(f"move 2 (mek → erk present): erk → pka gains {gain('pka', 'erk'):.1f}, pka → erk gains {gain('erk', 'pka', ('mek',)):.1f}")
print(f"p38 → pkc on its own gains {gain('pkc', 'p38'):.1f}; added after mek → pkc it gains {gain('pkc', 'p38', ('mek',)):.1f}")

🔍 **What just happened?** Starting from the empty graph, every step adds the edge with the largest BIC gain: first
the strongest dependences (Mek → Erk, Erk → PKA, Raf → Mek, …), then ever weaker ones, until after 16 moves no single
add, delete or reverse improves the score (BIC −16,381.4). Two things deserve attention. First, the score of a node
depends on its parents *jointly*: p38 → PKC would gain 564 points on its own, but once Mek → PKC is in it adds only
123 (printed above) — much of what p38 says about PKC is already said by Mek. Second, the very first arrow enters in
an arbitrary direction: on the empty graph, Mek → Erk and Erk → Mek gain exactly the same (891.65), because with BIC
both directions of a lone edge score the same (Section 10); the tie is broken by the last digits of floating-point
round-off, not by the data (here it happens to pick the true direction). Later choices are greedy, not ties: at move 2,
with Mek → Erk already present, Erk → PKA gains 741.0 but PKA → Erk only 93.2 (Erk's other parent Mek already explains
much of it), so the search picks Erk → PKA — against the true PKA → Erk. Whether the data can orient that edge at all
is a question about the *final* graph (Section 10).

**Problems with hill climbing.** It stops at the first **local maximum** — a graph whose one-edge changes all
lower the score but which is not the best overall — and it can wander on **plateaus** of equally scored graphs. The
standard remedies: **random restarts** (run again from random starting graphs and keep the best), **TABU search** (keep
a list of the $K$ most recently visited structures and forbid returning to them, which lets the search walk across
plateaus), and **simulated annealing** (sometimes accept a worse move, with a probability that decreases over time). We
try random restarts.

In [ ]:
# ── 9.4 Random restarts: different starting DAGs, different local maxima ──
def random_dag(rng, p_edge=0.2, max_parents=3):
    """A random DAG: random node order, each forward pair joined with probability p_edge."""
    order, G = rng.permutation(len(names)), [set() for _ in names]
    for a in range(len(names)):
        for b in range(a + 1, len(names)):
            if rng.random() < p_edge and len(G[order[b]]) < max_parents:
                G[order[b]].add(order[a])
    return G


rng = np.random.default_rng(SEED + 93)
runs = [(hist_hc, G_hc)]                                   # run 0: the empty-graph start of 9.3
for _ in range(20):
    G_r, h_r = hill_climb(random_dag(rng))
    runs.append((h_r, G_r))
finals = np.array([h[-1] for h, _ in runs])
best_hist, G_best = runs[int(np.argmax(finals))]
print("final BIC of the 21 runs:", np.round(np.sort(finals)[::-1], 1))
print(f"distinct local maxima: {len(np.unique(np.round(finals, 3)))}; best {finals.max():.1f}; from the empty graph {finals[0]:.1f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
for h, _ in runs[1:]:
    a1.plot(h, color="tab:blue", alpha=0.4, lw=1)
a1.plot(runs[0][0], color="tab:red", lw=2, label="start: empty graph")
a1.set_xlabel("hill-climbing move"); a1.set_ylabel("BIC"); a1.legend(); a1.set_title("score trajectories (21 runs)")
a1.set_ylim(finals.max() - 400, finals.max() + 40)
a2.hist(finals, bins=15, color="tab:blue")
a2.set_xlabel("final BIC (local maximum)"); a2.set_ylabel("runs"); a2.set_title("where the runs end")
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.5 🧪 The best network found, graded against the true network ──
best_edges = [(names[p], names[c]) for c in range(len(names)) for p in sorted(G_best[c])]
prec, rec, f1 = skeleton_scores(best_edges)
true_set = set(true_edges)
same_dir = [e for e in best_edges if e in true_set]
flipped = [e for e in best_edges if (e[1], e[0]) in true_set]
print(f"best DAG: {len(best_edges)} edges, BIC {bic_score(G_best):.1f} (true network: {bic_score(to_parent_sets(true_edges)):.1f})")
print(f"skeleton vs the true graph: precision {prec:.2f}, recall {rec:.2f}, F1 {f1:.2f}")
print(f"same direction as the truth ({len(same_dir)}):", same_dir)
print(f"reversed ({len(flipped)}):", flipped)
print("not in the true graph:", [e for e in best_edges if e not in same_dir and e not in flipped])
print("true edges missed:", [e for e in true_edges if frozenset(e) not in {frozenset(b) for b in best_edges}])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
draw_with_matches(axes[0], best_edges, f"best hill-climbing DAG (BIC), SIMULATED cells\ngreen = true edge (any direction)")
draw_graph(true_edges, SACHS_POS, ax=axes[1], radius=0.055, title="true network (20 edges)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 21 runs end at **21 different local maxima**, from −16,366.0 down to −16,668.2 — a
striking picture of the problem of local maxima: the empty-graph start (−16,381.4) is good but not the best, and a
single run proves nothing.

🧪 **Graded against the truth.** The best network has 17 edges and its skeleton is impressively right: 16 of its
edges are true edges (**precision 0.94**) and it finds 16 of the 20 true edges (**recall 0.80**). It misses PKC → Raf,
PIP3 → PLCγ, PKC → PKA and the almost invisible PIP2 → PKC, and adds one false edge, Erk → p38. The *directions* are
another story: only 8 of the 16 correct edges point the right way; 8 are reversed (Mek → Raf instead of Raf → Mek,
Erk → PKA instead of PKA → Erk, PIP2 → PLCγ instead of PLCγ → PIP2, …). Note also that the best learned network
scores *higher* than the true network (−16,366.0 vs −16,514.7): on discretised data the true graph's independences do
not hold exactly (Module 16 showed the leak), and a different graph can describe the three-level data better. A high
score is not a guarantee of truth.

✏️ **Your turn.** (a) Restrict `max_parents=2` and re-run `hill_climb` from the empty graph. Does the score change?
(b) Replace the BIC penalty `0.5 * np.log(N_CELLS)` by `1.0` (this is the Akaike criterion, AIC, after rescaling). How
many edges does the network have now, and how do precision and recall change? (Clear `_family_cache` first.)

### 📘 Lecture notes — General structure learning: hill climbing

- **Naïve approach: exhaustive search.** Score every structure and keep the best — impossible: the search space is
  exponentially large, and maintaining the DAG constraint is challenging. Use **heuristic search**.
- **Score.** Needs a complexity penalty, because the maximised likelihood always grows with more edges. BIC:
  $$\mathrm{BIC}(G)=\sum_i\Big[\sum_{j,k}N_{ijk}\log\frac{N_{ijk}}{N_{ij}}-\frac{(r_i-1)q_i}{2}\log N\Big].$$
  *In words:* each node's maximised log-likelihood given its parents, minus $\tfrac12\log N$ per free parameter. It is
  a sum over nodes, so one edge change alters at most two terms.
- **Hill climbing.** Start with an initial structure. Repeat: generate neighbours by **adding**, **deleting** or
  **reversing** one edge (adds and reversals that create a directed cycle are not permitted — e.g. adding D→B to
  A→C←B, C→D); score them; move to the best. Terminate when no neighbour improves the score; return the network.
- **Problems.** **Local maxima** (every one-edge change lowers the score, yet the graph is not optimal) and
  **plateaus** (neighbours with equal scores). **Remedies:** random restarts; TABU search (keep a list of the $K$
  most recently visited structures and avoid them — helps to cross plateaus); simulated annealing.

> 🎓 **Exam tip.** Know why the score must penalise complexity (the maximum likelihood always increases with more edges)
    and which moves are illegal (an add or reverse that creates a directed cycle, e.g. adding D→B when B→C→D
    exists).

---
## 10 · What observational data cannot tell you

> 📍 **Where we are.** Section 9 found half of the arrows pointing against the true network. Is the algorithm wrong?
> Partly the data simply cannot decide. This section explains why — the most important caveat of the whole
> Bayesian-network trilogy.

**Markov equivalence.** The three graphs $X\to Y\to Z$, $X\leftarrow Y\leftarrow Z$ and $X\leftarrow Y\to Z$ imply the
*same* independences (only $X\perp Z\mid Y$; Module 16) and can represent exactly the same joint distributions. No
amount of observational data can tell them apart. Only the collider $X\to Y\leftarrow Z$ is different ($X\perp Z$, but
not given $Y$). The general theorem (Verma & Pearl 1990): **two DAGs are Markov equivalent if and only if they have the
same skeleton and the same v-structures** (colliders whose two parents are not adjacent). Observational data can at
best identify the equivalence class: the skeleton, the v-structures, and the arrows that these force.

Scores such as BIC respect this: equivalent graphs get identical scores. Let us check it on the networks we learned.

In [ ]:
# ── 10.1 BIC cannot see the direction of many edges ──
print("Chow–Liu tree rooted at each protein — BIC:")
scores_by_root = {r: bic_score(to_parent_sets(orient_tree([tuple(e) for e in cl_sachs], r))) for r in names}
print("  " + ", ".join(f"{r}: {s:.3f}" for r, s in scores_by_root.items()))
print(f"  spread across roots: {max(scores_by_root.values()) - min(scores_by_root.values()):.1e}")

print("\nreversing each edge of the best DAG (if still acyclic):")
base = bic_score(G_best)
for a, b in best_edges:
    i, j = names.index(a), names.index(b)
    G = [set(pa) for pa in G_best]
    G[j].discard(i)
    if reaches(G, i, j):
        print(f"  {a:>4s} → {b:<4s}: reversal would create a cycle")
        continue
    G[i].add(j)
    delta = bic_score(G) - base
    covered = G_best[j] == G_best[i] | {i}                 # child's parents = parent's parents + parent
    print(f"  {a:>4s} → {b:<4s}: ΔBIC = {delta:+9.3f}  "
          f"{'(covered edge: direction not identifiable by this score)' if abs(delta) < 1e-6 and covered else ''}")

🔍 **What just happened?** The Chow–Liu tree gets *exactly* the same BIC whichever protein is chosen as root: all
eleven orientations are Markov equivalent (a tree has no colliders). For the best hill-climbing DAG, two arrows — Mek →
Erk and Erk → PKA — can be reversed without changing the BIC at all. Both are **covered** edges in the final DAG (for
Erk → PKA: PKA's parents are exactly Erk's parents plus Erk, namely {Mek, Erk}), so reversing either gives a Markov
equivalent graph: the data cannot orient them. Erk → PKA, which points against the truth, was chosen greedily during
the search; in the final graph no observational score could have told the two directions apart.
Six other reversals would create a directed cycle. The remaining nine change the score by 35–193 points: those
arrows belong to (or would create) v-structures, so their direction is "supported" by an independence pattern — and
several of them are nevertheless wrong. The clearest case is PIP2 → PLCγ ← PKC (reversing PIP2 → PLCγ costs 193 points),
a collider at PLCγ although the truth is PLCγ → PIP2 and PLCγ → PKC. Why? PIP2 and PKC are truly connected (PIP2 → PKC),
but their routes cancel almost exactly, so in the data they look independent — and "independent, but dependent given
PLCγ" is the signature of a collider. A supported direction is only as trustworthy as the assumptions behind it
(faithfulness, no hidden common causes, Module 16) and the sample it was learned from.

How much could observational data reveal *at best*? Two DAGs are equivalent if one can be turned into the other by
reversing **covered** edges one at a time (an edge X → Y is covered when Y's parents are exactly X's parents plus X;
Chickering 1995). Starting from the true network and reversing covered edges in every possible way enumerates its whole
equivalence class.

In [ ]:
# ── 10.2 🧪 The Markov equivalence class of the true signalling network ──
def equivalence_class(edges):
    """All DAGs Markov equivalent to `edges`, found by repeatedly reversing covered edges (breadth-first)."""
    start = frozenset(edges)
    seen, queue = {start}, [start]
    while queue:
        E = queue.pop(0)
        pa = {v: {a for a, b in E if b == v} for v in names}
        for a, b in E:
            if pa[b] == pa[a] | {a}:                            # a → b is covered: reversing it keeps equivalence
                F = frozenset((E - {(a, b)}) | {(b, a)})
                if F not in seen:
                    seen.add(F); queue.append(F)
    return seen


eq_class = equivalence_class(true_edges)
compelled = [e for e in true_edges if all(e in E for E in eq_class)]
pa_true = {v: {a for a, b in true_edges if b == v} for v in names}
v_structs = [(a, c, b) for c in names for a, b in itertools.combinations(sorted(pa_true[c]), 2)
             if (a, b) not in true_edges and (b, a) not in true_edges]
print(f"DAGs equivalent to the true network: {len(eq_class)}")
print("v-structures of the true network:", [f"{a}→{c}←{b}" for a, c, b in v_structs])
print(f"arrows with the same direction in all {len(eq_class)} equivalent DAGs: {compelled}")
print(f"→ {len(true_edges) - len(compelled)} of the {len(true_edges)} true arrows cannot be oriented from observational data, "
      "even with infinitely many cells")

🔍 **What just happened?** 🧪 The true network is one of **176** DAGs that imply exactly the same conditional
independences. Its only v-structures are the two colliders at Akt (PIP3 → Akt ← Erk and PIP3 → Akt ← PKA), and only
the **3 arrows into Akt** point the same way in all 176 graphs. The other 17 directions — including Raf → Mek, Mek →
Erk and PKA → Erk — cannot be learned from unperturbed cells at all, however many we measure. The reversed arrows of
Section 9 are therefore not a bug in hill climbing: observational data do not contain that information.

**How Sachs *et al.* oriented their network.** They did not rely on observational data alone: their measurements
include cells in which specific molecules were inhibited or activated (the simulator reproduces this design:
perturbations of PKC, PKA, Akt, PIP2 and Mek). An **intervention** cuts the arrows *into* the perturbed node: if
inhibiting Mek changes Erk but inhibiting Erk does not change Mek, the arrow is Mek → Erk. Interventions break Markov
equivalence; that is why perturbation experiments are central to network inference in biology —
[Module 41](41_perturbation_causal_inference.ipynb) develops this into causal inference from perturbation data.

✏️ **Your turn.** Score the four three-node graphs raf → mek → erk, raf ← mek ← erk, raf ← mek → erk and
raf → mek ← erk with `bic_score(to_parent_sets([...]))`. Which of them tie, and why? What does the collider's score
tell you about these data?

⚠️ **A learned edge is a statistical dependence, not a mechanism.** Even a perfectly learned Bayesian network says
"these variables are dependent in a way not explained by the other measured variables". The dependence can come from a
direct biochemical reaction, from an unmeasured intermediate or common regulator (in real cells: dozens of unmeasured
kinases), from discretisation (the false edge Erk → p38 above), or from mixing heterogeneous cell populations.
Feedback loops (Erk ⊣ Raf) cannot be drawn at all in a DAG. Treat a learned network as a set of **hypotheses to test**
— with perturbations, as Sachs *et al.* did.

### 📘 Lecture notes — Bayesian networks: what you should know

- **Representation.** A Bayesian network represents a joint distribution as a DAG + conditional distributions,
  $P(X_1,\dots,X_n)=\prod_iP(X_i\mid\mathrm{Pa}(X_i))$ — far fewer parameters than the full table (Module 15).
  Markov blankets and d-separation decode the conditional-independence assumptions (Module 16).
- **Inference.** NP-hard in general; for some graphs (chains, polytrees) efficient exact inference is feasible.
  Enumeration (exact, slow), variable elimination (exact, faster; 19 → 14 operations in the worked example),
  stochastic methods (approximate, fast; $N_B/N_c$).
- **Learning.** Easy for a known graph with fully observed data: MLE = conditional frequencies, MAP = with
  pseudo-counts. Learning the graph: Chow–Liu finds the best tree exactly (maximum spanning tree on mutual
  informations); for general DAGs, heuristic score-based search (hill climbing with BIC, restarts, TABU,
  annealing). Hardest when the graph is unknown.
- **Naïve Bayes revisited.** The full joint $P(X_1,\dots,X_n,Y)$ needs exponentially many parameters; the naïve
  Bayes assumption (features independent given $Y$) is the BN $Y\to X_j$ with a linear number.
- **What data can and cannot say.** Observational data identify at most the Markov equivalence class (skeleton +
  v-structures); interventions are needed to orient the rest.

> 🎓 **Exam tip.** Typical exam questions: count the parameters of a network; read off a Markov blanket; decide a d-separation
    query listing all paths; compute a conditional probability by enumeration and count the operations; write the MLE of
    a CPT entry as a ratio of counts; run Kruskal on a small weighted graph; explain why a score needs a complexity
    penalty and why hill climbing needs restarts.

---
## 📌 Key takeaways

- **Representation → inference → learning.** Questions involving some variables need sums over the others: query,
  evidence and hidden variables. $P(B\mid J,\neg M)=0.0063175/0.0555=0.114$ (rounding each term first gives 0.00601).
- **Enumeration** is exact but exponential in the number of hidden variables (19 operations here, doubling per hidden
  variable). **Variable elimination** pushes sums inside products and reuses factors (19 → 14 here; linear instead of
  exponential on chains and polytrees). Exact inference is **NP-hard** in general.
- **Stochastic inference**: ancestral sampling + counting, $P(B\mid J,\neg M)\approx N_B/N_c$; error $\propto1/\sqrt{N_c}$;
  rejection sampling starves when the evidence is rare.
- **Learning CPTs with a known graph** is easy: the log-likelihood decomposes and each CPT entry's MLE is a conditional
  frequency; pseudo-counts (MAP) protect rare parent configurations.
- **Learning the graph** is hard ($3.2\times10^{22}$ DAGs on 11 nodes). **Chow–Liu** finds the best tree exactly
  (maximum spanning tree on mutual informations ⇔ minimum KL); **hill climbing with BIC** searches general DAGs but stops
  at local maxima (random restarts, TABU, annealing).
- 🧪 On **simulated Sachs-like cells** with a known true graph, the Chow–Liu tree has 9 of 10 edges right (precision
  0.90) and the best of 21 hill-climbing runs finds 16 of the 20 true edges with one false edge (precision 0.94,
  recall 0.80) — but only half of its arrows point the right way, and nearly cancelling routes (PIP2 → PKC) hide true
  edges and fake colliders.
- **Observational data identify only a Markov equivalence class**: the true signalling network has 176 equivalent DAGs
  and only 3 of its 20 arrows are identifiable; interventions are needed to orient the rest, and a learned edge is a
  dependence, not a proven mechanism.

---
## ✅ Checkpoint

1. Using the alarm CPTs, write $P(\neg B, J,\neg M)$ as a sum of four products and identify which term is largest.
   <details><summary>Answer</summary>$\sum_{a,e}P(\neg B)P(e)P(a\mid\neg B,e)P(J\mid a)P(\neg M\mid a)$; the largest term is $a=0,e=0$: $0.95\times0.9\times0.95\times0.05\times0.85\approx0.0345$ — a false call by John on a quiet day.</details>
2. How many terms does enumeration need for $P(J\mid B)$ in the alarm network, and how many multiplications per term?
   <details><summary>Answer</summary>Hidden variables: E, A, M → $2^3=8$ terms per value of J, each a product of 5 factors (4 multiplications). (With VE one would first notice that summing out the leaf M contributes a factor 1.)</details>
3. Rewrite $\sum_e\sum_a P(B)P(e)P(a\mid B,e)P(J\mid a)P(\neg M\mid a)$ with the sums pushed inside, and explain why this is cheaper.
   <details><summary>Answer</summary>$P(B)\sum_e P(e)\sum_a P(a\mid B,e)P(J\mid a)P(\neg M\mid a)$: factors that do not depend on a summation variable are multiplied once instead of once per term (distributive law); 14 instead of 19 operations.</details>
4. You draw $10^4$ samples and 555 satisfy $J=1, M=0$, of which 63 have $B=1$. Estimate $P(B\mid J,\neg M)$ and its standard error.
   <details><summary>Answer</summary>$63/555\approx0.114$; standard error $\sqrt{0.114\times0.886/555}\approx0.013$ — it depends on $N_c=555$, not on $N=10^4$.</details>
5. From 1,000 samples you count 12 days with $B=1,E=0$, of which 10 have $A=1$. Give the MLE and the MAP estimate with α = 1 of $P(A{=}1\mid B{=}1,E{=}0)$.
   <details><summary>Answer</summary>MLE $10/12\approx0.833$; MAP $(10+1)/(12+2)=11/14\approx0.786$ (true value 0.85).</details>
6. Why does maximising the sum of mutual informations over a tree's edges minimise $\mathrm{KL}(P\|T)$?
   <details><summary>Answer</summary>$\mathrm{KL}(P\|T)=-H(P)+\sum_i H(X_i)-\sum_{\text{edges}}I(X_i,X_j)$ when T uses the true conditionals; only the last term depends on the tree, with a minus sign.</details>
7. Kruskal's algorithm on the seven-node worked example rejects B–C (weight 1/8). Why, although 1/8 is larger than the weight of E–G, which is accepted?
   <details><summary>Answer</summary>When B–C is considered, B and C are already connected (B–E–C), so adding B–C would close a cycle; E–G joins G, which is not yet connected to the tree.</details>
8. A hill-climbing search returns Mek → Raf, although biology says Raf → Mek. Give two reasons this can happen with observational data.
   <details><summary>Answer</summary>(1) Markov equivalence: reversing the edge may leave the BIC unchanged, so the data cannot choose a direction (the search order decides) — in the true signalling network only the 3 arrows into Akt are identifiable. (2) Hidden variables, discretisation, nearly cancelling routes or a local maximum can favour the wrong direction. Interventions (inhibiting Mek or Raf) are needed to orient it.</details>

## 🧠 Questions & discussion

1. Variable elimination's cost depends on the elimination order. Propose a simple greedy rule for choosing the next variable to eliminate, and test it on the Sachs consensus network.
2. Rejection sampling wastes samples when the evidence is rare. In a single-cell experiment, every cell's full measurement vector is 'rare'. How does Gibbs sampling (Module 21) avoid this problem?
3. PIP2 → PKC is a true edge of the simulated network, yet its mutual information is almost zero. With real data, how would you tell an absent edge from a present but cancelled or uninformative one?
4. BIC penalises each parameter by ½ log N. With 100,000 cells instead of 1,800, would you expect more or fewer edges? Would the arrow-direction problem of Section 10 go away?
5. Design a minimal set of perturbation experiments that would orient the edges among Raf, Mek and Erk. What would you measure in each?

## 🚀 Bonus — going further

- **Use the interventions.** Load `simulate_signaling_cytometry('discrete', observational_only=False)`. For cells with a perturbation of protein X, leave X's CPT out of the BIC (its value was set by the experimenter) and re-run hill climbing on all 5,400 cells. Do the arrow directions agree better with the true network? ([Module 41](41_perturbation_causal_inference.ipynb) develops learning from perturbations.)
- **Likelihood weighting.** Implement likelihood weighting for $P(B\mid J{=}1,M{=}1,E{=}1,A{=}0)$ (fix the evidence; weight each sample by the product of the evidence CPT entries) and compare its accuracy with rejection sampling for the same number of samples.
- **Continuous data.** Use `simulate_signaling_cytometry('continuous')` and a linear-Gaussian BIC (residual variance of each protein regressed on its parents, Module 06) for hill climbing. Compare with Module 18's graphical lasso.
- **Exact search for small n.** For the alarm samples (5 variables, 29,281 DAGs), score every DAG with BIC and check whether hill climbing finds the global maximum. How many DAGs tie with the best score (the size of its equivalence class)?
- **Related notebooks in this repository.** [`../../FundamentalsLearningUncertainty/`](../../FundamentalsLearningUncertainty/) (MLE and MAP, the basis of Section 5) and [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/) (a generative Bayesian-network classifier).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 17 — Bayesian networks III: inference and learning     date:
Input:            alarm network (classic CPTs) and samples from it; SIMULATED Sachs-like discrete data
                  (1,800 unperturbed cells × 11 proteins) generated from the 20-edge consensus network
Output:           P(B,J,¬M) = 0.0063175, P(B|J,¬M) = 0.114; 19 vs 14 operations; sampling error vs N;
                  learned CPTs vs N; Chow–Liu tree and BIC hill-climbing DAG of the simulated cells with
                  precision/recall against the true graph; its Markov equivalence class (176 DAGs)
Method:           enumeration; variable elimination with factors; ancestral + rejection sampling; MLE/MAP by
                  counting; Chow–Liu (mutual information + Kruskal); hill climbing with BIC and restarts
Key equations:    P(q|e) ∝ Σ_h Π_i P(x_i|pa_i);  θ̂ = N(x, pa)/N(pa);  KL(P||T) = const − Σ_edges I;
                  BIC = Σ_i [loglik_i − (r_i − 1) q_i log(N)/2]
Key parameters:   elimination order; number of samples; pseudo-count α; root of the tree; max parents;
                  number of restarts; BIC penalty
What changed when I modified the root / α / the penalty / the starting graph:
One failure mode: arrows in a learned network point the wrong way because observational data cannot orient them
Connection to the 02-620 topics (which idea of the lecture this illustrates):
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the third part of its lecture on Bayesian networks (inference and learning).
- Koller D., Friedman N. (2009) *Probabilistic Graphical Models: Principles and Techniques*. MIT Press — Chapters 9 (variable elimination), 12 (sampling), 17–18 (parameter and structure learning).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapters 20 (exact inference) and 26 (graphical model structure learning).
- Sachs K., Perez O., Pe'er D., Lauffenburger D. A., Nolan G. P. (2005) Causal protein-signaling networks derived from multiparameter single-cell data. *Science* 308:523–529.
- Chow C. K., Liu C. N. (1968) Approximating discrete probability distributions with dependence trees. *IEEE Transactions on Information Theory* 14:462–467.
- Kruskal J. B. (1956) On the shortest spanning subtree of a graph and the traveling salesman problem. *Proceedings of the American Mathematical Society* 7:48–50.
- Schwarz G. (1978) Estimating the dimension of a model. *Annals of Statistics* 6:461–464.
- Cooper G. F. (1990) The computational complexity of probabilistic inference using Bayesian belief networks. *Artificial Intelligence* 42:393–405.
- Robinson R. W. (1973) Counting labeled acyclic digraphs. In F. Harary (ed.), *New Directions in the Theory of Graphs*, Academic Press, pp. 239–273.
- Verma T., Pearl J. (1990) Equivalence and synthesis of causal models. *Proceedings of the 6th Conference on Uncertainty in Artificial Intelligence*, pp. 220–227.
- Chickering D. M. (1995) A transformational characterization of equivalent Bayesian network structures. *Proceedings of the 11th Conference on Uncertainty in Artificial Intelligence*, pp. 87–98.
- Heckerman D., Geiger D., Chickering D. M. (1995) Learning Bayesian networks: the combination of knowledge and statistical data. *Machine Learning* 20:197–243.

## ➡️ Next

Bayesian networks need arrows, and arrows are exactly what observational data struggle to provide. In
[Module 18 · Gaussian graphical models: networks from correlations](18_gaussian_graphical_models.ipynb) you will
drop the arrows altogether, learn an *undirected* network from continuous (simulated) signalling measurements with the graphical
lasso, and compare it with what you found here.